# 1. Installing Libraries 

In [25]:
#Checking Python & pip version first
import sys
print("Python version:", sys.version)
!pip --version

Python version: 3.11.9 | packaged by conda-forge | (main, Apr 19 2024, 18:36:13) [GCC 12.3.0]
pip 24.0 from /opt/conda/lib/python3.11/site-packages/pip (python 3.11)


In [26]:
#checking what already there 
!pip list | grep -E "torch|transformers|cuda|peft|trl|bitsandbytes|wandb|datasets|accelerate"

accelerate                   0.33.0
bitsandbytes                 0.43.3
datasets                     2.20.0
nvidia-cuda-cupti-cu12       12.1.105
nvidia-cuda-nvcc-cu12        12.3.107
nvidia-cuda-nvrtc-cu12       12.1.105
nvidia-cuda-runtime-cu12     12.1.105
peft                         0.12.0
torch                        2.5.1+cu121
torchaudio                   2.5.1+cu121
torchvision                  0.20.1+cu121
transformers                 4.44.0
trl                          0.9.6
wandb                        0.17.0


In [27]:
#installing pytorch
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet

In [28]:
#verifying pytorch + GPU
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

total_mem = torch.cuda.get_device_properties(0).total_memory
print("GPU Memory:", round(total_mem / 1e9, 1), "GB")

PyTorch: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
GPU: NVIDIA L40
GPU Memory: 47.7 GB


In [29]:
#hugging face ecosystem
!pip install transformers==4.44.0 datasets==2.20.0 huggingface_hub==0.24.0 tokenizers==0.19.1 accelerate==0.33.0 --quiet
print("HuggingFace libraries installed!")

HuggingFace libraries installed!


In [30]:
#QLoRA 
!pip install bitsandbytes==0.43.3 peft==0.12.0 trl==0.9.6 --quiet
print("QLoRA libraries installed!")

QLoRA libraries installed!


In [31]:
#tracking and utilities
!pip install wandb==0.17.0 scipy sentencepiece protobuf sqlparse sqlglot scikit-learn --quiet
print("Utilities installed!")

Utilities installed!


In [32]:
import transformers, datasets, peft, trl, bitsandbytes, wandb, sqlparse

print("transformers:", transformers.__version__)
print("datasets:    ", datasets.__version__)
print("peft:        ", peft.__version__)
print("trl:         ", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("wandb:       ", wandb.__version__)
print("sqlparse:    ", sqlparse.__version__)
print("\nAll libraries ready!")

transformers: 4.44.0
datasets:     2.20.0
peft:         0.12.0
trl:          0.9.6
bitsandbytes: 0.43.3
wandb:        0.17.0
sqlparse:     0.5.5

All libraries ready!


In [43]:
#logging to Hugging Face
from huggingface_hub import login

HF_TOKEN = "Your Key Here"  

login(token=HF_TOKEN, add_to_git_credential=True)
print("HuggingFace login successful!")

Token is valid (permission: write).
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
Token has not been saved to git credential helper.
Your token has been saved to /home/jovyan/.cache/huggingface/token
Login successful
HuggingFace login successful!


# 2. Dataset Creation


In [17]:
#loading spider dataset
from datasets import load_dataset

print("Loading Spider dataset....")
spider = load_dataset("spider")

print("\nSpider loaded successfully!")
print("Train examples:", len(spider['train']))
print("Validation examples:", len(spider['validation']))
print("\nFirst example keys:", list(spider['train'][0].keys()))

Loading Spider dataset....


Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]


Spider loaded successfully!
Train examples: 7000
Validation examples: 1034

First example keys: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks']


In [18]:
#The model already knows your exact table and column names → generates perfect SQL every time so for that reason we need table schema.
import urllib.request
import zipfile
import os

print("Downloading Spider schema files...")

# Download official Spider zip
url = "https://drive.usercontent.google.com/download?id=1TqleXec_OykOYFREKKtschzY29dUcVAQ&export=download&confirm=t"
zip_path = "spider.zip"

urllib.request.urlretrieve(url, zip_path)
print("Downloaded!")

# Extract only what we need
print("Extracting schema files...")
with zipfile.ZipFile(zip_path, 'r') as z:
    # We only need tables.json and database folder
    for file in z.namelist():
        if 'tables.json' in file or 'database' in file:
            z.extract(file, '.')

print("Extracted!")
print("Files available:", os.listdir('spider'))

Downloaded!
Extracting schema files...
Extracted!
Files available: ['database', 'tables.json']


In [19]:
#exploringtable schema file of spider
import json

# Load tables.json
with open('spider/tables.json') as f:
    tables = json.load(f)

print("=" * 60)
print("SPIDER SCHEMA FILE OVERVIEW")
print("=" * 60)
print(f"\nTotal databases: {len(tables)}")

# Show structure of first database
sample_db = tables[0]
print(f"\nSAMPLE DATABASE: '{sample_db['db_id']}'")
print(f"\nKeys in each database entry:")
for key in sample_db.keys():
    print(f"  → {key}")

print(f"\nTABLE NAMES:")
for t in sample_db['table_names_original']:
    print(f"  → {t}")

print(f"\nCOLUMN NAMES (table_index, column_name):")
for col in sample_db['column_names_original']:
    print(f"  → {col}")

print(f"\nCOLUMN TYPES:")
for i, col in enumerate(sample_db['column_names_original']):
    print(f"  → {col[1]} : {sample_db['column_types'][i]}")

SPIDER SCHEMA FILE OVERVIEW

Total databases: 166

SAMPLE DATABASE: 'perpetrator'

Keys in each database entry:
  → column_names
  → column_names_original
  → column_types
  → db_id
  → foreign_keys
  → primary_keys
  → table_names
  → table_names_original

TABLE NAMES:
  → perpetrator
  → people

COLUMN NAMES (table_index, column_name):
  → [-1, '*']
  → [0, 'Perpetrator_ID']
  → [0, 'People_ID']
  → [0, 'Date']
  → [0, 'Year']
  → [0, 'Location']
  → [0, 'Country']
  → [0, 'Killed']
  → [0, 'Injured']
  → [1, 'People_ID']
  → [1, 'Name']
  → [1, 'Height']
  → [1, 'Weight']
  → [1, 'Home Town']

COLUMN TYPES:
  → * : text
  → Perpetrator_ID : number
  → People_ID : number
  → Date : text
  → Year : number
  → Location : text
  → Country : text
  → Killed : number
  → Injured : number
  → People_ID : number
  → Name : text
  → Height : number
  → Weight : number
  → Home Town : text


In [20]:
def build_schema(db_id, tables_data):
    """
    Converted tables.json into clean 
    CREATE TABLE statementsdef build_schema(db_id, tables_data):
    """
    # Find the database entry
    db = next((d for d in tables_data if d['db_id'] == db_id), None)
    if db is None:
        return ""
    
    create_statements = []
    
    # Loop through each table
    for table_idx, table_name in enumerate(db['table_names_original']):
        
        # Get all columns belonging to this table
        columns = []
        for col_info, col_type in zip(
            db['column_names_original'], 
            db['column_types']
        ):
            # col_info = [table_index, column_name]
            # Skip wildcard (*) and columns from other tables
            if col_info[0] == table_idx:
                col_name = col_info[1].replace(' ', '_')
                col_type = col_type.upper()
                columns.append(f"  {col_name} {col_type}")
        
        # Build CREATE TABLE statement
        if columns:
            stmt = f"CREATE TABLE {table_name} (\n"
            stmt += ",\n".join(columns)
            stmt += "\n);"
            create_statements.append(stmt)
    
    return "\n\n".join(create_statements)

# Test it on our sample database
schema = build_schema('perpetrator', tables)
print("=" * 60)
print("CONVERTED SCHEMA:")
print("=" * 60)
print(schema)

# Test on another database
print("\n" + "=" * 60)
print("ANOTHER EXAMPLE - concert_singer:")
print("=" * 60)
print(build_schema('concert_singer', tables))

CONVERTED SCHEMA:
CREATE TABLE perpetrator (
  Perpetrator_ID NUMBER,
  People_ID NUMBER,
  Date TEXT,
  Year NUMBER,
  Location TEXT,
  Country TEXT,
  Killed NUMBER,
  Injured NUMBER
);

CREATE TABLE people (
  People_ID NUMBER,
  Name TEXT,
  Height NUMBER,
  Weight NUMBER,
  Home_Town TEXT
);

ANOTHER EXAMPLE - concert_singer:
CREATE TABLE stadium (
  Stadium_ID NUMBER,
  Location TEXT,
  Name TEXT,
  Capacity NUMBER,
  Highest NUMBER,
  Lowest NUMBER,
  Average NUMBER
);

CREATE TABLE singer (
  Singer_ID NUMBER,
  Name TEXT,
  Country TEXT,
  Song_Name TEXT,
  Song_release_year TEXT,
  Age NUMBER,
  Is_male OTHERS
);

CREATE TABLE concert (
  concert_ID NUMBER,
  concert_Name TEXT,
  Theme TEXT,
  Stadium_ID TEXT,
  Year TEXT
);

CREATE TABLE singer_in_concert (
  concert_ID NUMBER,
  Singer_ID TEXT
);


In [21]:
import json
import pandas as pd
from datasets import Dataset

# Load schemas
with open('spider/tables.json') as f:
    tables = json.load(f)

# Step 1: Build schema map (db_id -> CREATE TABLE string)
# Combined schema + question + SQL into one complete training example

def build_schema(db):
    statements = []
    for i, table_name in enumerate(db['table_names_original']):
        cols = []
        for col in db['column_names_original']:
            if col[0] == i:  # col[0] is table index
                col_name = col[1]
                col_idx  = db['column_names_original'].index(col)
                col_type = db['column_types'][col_idx]
                cols.append(f"  {col_name} {col_type}")
        statement = f"CREATE TABLE {table_name} (\n" + ",\n".join(cols) + "\n);"
        statements.append(statement)
    return "\n\n".join(statements)

# Build map for all 200 databases
schema_map = {}
for db in tables:
    schema_map[db['db_id']] = build_schema(db)

print(f"Schema map built for {len(schema_map)} databases")

# Step 2: Format each Spider example into prompt
PROMPT_TEMPLATE = """### Instruction:
You are an expert SQL assistant. Given a database schema and a natural language question, write a correct SQL query. Only output the SQL query, no explanation.

### Schema:
{schema}

### Question:
{question}

### SQL:
{sql}"""

def format_spider_example(example):
    schema = schema_map.get(example['db_id'], "")
    if not schema:
        return None
    return {
        "text": PROMPT_TEMPLATE.format(
            schema=schema,
            question=example['question'],
            sql=example['query']
        ),
        "db_id"     : example['db_id'],
        "question"  : example['question'],
        "sql"       : example['query'],
        "source"    : "spider"
    }

# Step 3: Apply to all Spider train examples
print("\nFormatting Spider training examples...")
spider_formatted = []
for ex in spider['train']:
    formatted = format_spider_example(ex)
    if formatted:
        spider_formatted.append(formatted)

print(f"Formatted {len(spider_formatted):,} Spider examples")

# Step 4: Show one complete formatted example
print("\n" + "=" * 60)
print("COMPLETE TRAINING EXAMPLE (what model will learn from):")
print("=" * 60)
print(spider_formatted[0]['text'])
print("\n" + "=" * 60)
print(f"Total characters in this example: {len(spider_formatted[0]['text'])}")

Schema map built for 166 databases

Formatting Spider training examples...
Formatted 7,000 Spider examples

COMPLETE TRAINING EXAMPLE (what model will learn from):
### Instruction:
You are an expert SQL assistant. Given a database schema and a natural language question, write a correct SQL query. Only output the SQL query, no explanation.

### Schema:
CREATE TABLE department (
  Department_ID number,
  Name text,
  Creation text,
  Ranking number,
  Budget_in_Billions number,
  Num_Employees number
);

CREATE TABLE head (
  head_ID number,
  name text,
  born_state text,
  age number
);

CREATE TABLE management (
  department_ID number,
  head_ID number,
  temporary_acting text
);

### Question:
How many heads of the departments are older than 56 ?

### SQL:
SELECT count(*) FROM head WHERE age  >  56

Total characters in this example: 647


In [35]:
#  exact bike-share schema 
BIKESHARE_SCHEMA = """CREATE TABLE trips (
  trip_id INTEGER,
  start_time TIMESTAMP,
  end_time TIMESTAMP,
  duration_minutes FLOAT,
  start_station_id VARCHAR,
  end_station_id VARCHAR,
  bike_id VARCHAR,
  bike_type VARCHAR,
  passholder_type VARCHAR
);

CREATE TABLE stations (
  station_id VARCHAR,
  station_name VARCHAR,
  lat FLOAT,
  lon FLOAT
);

CREATE TABLE weather (
  timestamp TIMESTAMP,
  temp_c FLOAT,
  precip_mm FLOAT,
  wind_speed_mps FLOAT,
  rel_humidity FLOAT,
  pressure_hpa FLOAT,
  weather_code INTEGER
);"""

# 300 bike-share pairs (100 easy, 100 medium, 100 hard)
bikeshare_pairs = [

    # ========== EASY (100 pairs) ==========
    # Simple SELECT, COUNT, WHERE, basic filters

    {"question": "How many trips were taken in total?",
     "sql": "SELECT COUNT(*) FROM trips;"},

    {"question": "How many electric bike trips were there?",
     "sql": "SELECT COUNT(*) FROM trips WHERE bike_type = 'Electric';"},

    {"question": "How many standard bike trips were there?",
     "sql": "SELECT COUNT(*) FROM trips WHERE bike_type = 'Standard';"},

    {"question": "What is the average trip duration in minutes?",
     "sql": "SELECT AVG(duration_minutes) FROM trips;"},

    {"question": "What is the maximum trip duration?",
     "sql": "SELECT MAX(duration_minutes) FROM trips;"},

    {"question": "What is the minimum trip duration?",
     "sql": "SELECT MIN(duration_minutes) FROM trips;"},

    {"question": "How many trips were taken in July?",
     "sql": "SELECT COUNT(*) FROM trips WHERE MONTH(start_time) = 7;"},

    {"question": "How many trips were taken in January?",
     "sql": "SELECT COUNT(*) FROM trips WHERE MONTH(start_time) = 1;"},

    {"question": "How many trips were taken in December?",
     "sql": "SELECT COUNT(*) FROM trips WHERE MONTH(start_time) = 12;"},

    {"question": "How many trips lasted more than 60 minutes?",
     "sql": "SELECT COUNT(*) FROM trips WHERE duration_minutes > 60;"},

    {"question": "How many trips lasted less than 10 minutes?",
     "sql": "SELECT COUNT(*) FROM trips WHERE duration_minutes < 10;"},

    {"question": "How many trips were taken on electric bikes in July?",
     "sql": "SELECT COUNT(*) FROM trips WHERE bike_type = 'Electric' AND MONTH(start_time) = 7;"},

    {"question": "What is the total number of stations?",
     "sql": "SELECT COUNT(*) FROM stations;"},

    {"question": "What are all the unique bike types?",
     "sql": "SELECT DISTINCT bike_type FROM trips;"},

    {"question": "What are all the unique passholder types?",
     "sql": "SELECT DISTINCT passholder_type FROM trips;"},

    {"question": "How many monthly pass trips were taken?",
     "sql": "SELECT COUNT(*) FROM trips WHERE passholder_type = 'Monthly Pass';"},

    {"question": "How many walk-up trips were taken?",
     "sql": "SELECT COUNT(*) FROM trips WHERE passholder_type = 'Walk-up';"},

    {"question": "What is the average temperature in the weather data?",
     "sql": "SELECT AVG(temp_c) FROM weather;"},

    {"question": "How many hours had precipitation greater than zero?",
     "sql": "SELECT COUNT(*) FROM weather WHERE precip_mm > 0;"},

    {"question": "What is the maximum temperature recorded?",
     "sql": "SELECT MAX(temp_c) FROM weather;"},

    {"question": "What is the minimum temperature recorded?",
     "sql": "SELECT MIN(temp_c) FROM weather;"},

    {"question": "How many trips were taken in the morning between 6am and 12pm?",
     "sql": "SELECT COUNT(*) FROM trips WHERE HOUR(start_time) BETWEEN 6 AND 11;"},

    {"question": "How many trips were taken in the evening between 5pm and 9pm?",
     "sql": "SELECT COUNT(*) FROM trips WHERE HOUR(start_time) BETWEEN 17 AND 21;"},

    {"question": "How many trips were taken at night after 9pm?",
     "sql": "SELECT COUNT(*) FROM trips WHERE HOUR(start_time) >= 21;"},

    {"question": "How many trips were taken on weekends?",
     "sql": "SELECT COUNT(*) FROM trips WHERE DAYOFWEEK(start_time) IN (1, 7);"},

    {"question": "How many trips were taken on weekdays?",
     "sql": "SELECT COUNT(*) FROM trips WHERE DAYOFWEEK(start_time) BETWEEN 2 AND 6;"},

    {"question": "How many trips started and ended at the same station?",
     "sql": "SELECT COUNT(*) FROM trips WHERE start_station_id = end_station_id;"},

    {"question": "What is the total trip duration in minutes?",
     "sql": "SELECT SUM(duration_minutes) FROM trips;"},

    {"question": "How many trips were taken in summer?",
     "sql": "SELECT COUNT(*) FROM trips WHERE MONTH(start_time) IN (6, 7, 8);"},

    {"question": "How many trips were taken in winter?",
     "sql": "SELECT COUNT(*) FROM trips WHERE MONTH(start_time) IN (12, 1, 2);"},

    {"question": "How many trips were taken on Mondays?",
     "sql": "SELECT COUNT(*) FROM trips WHERE DAYOFWEEK(start_time) = 2;"},

    {"question": "How many trips were taken on Fridays?",
     "sql": "SELECT COUNT(*) FROM trips WHERE DAYOFWEEK(start_time) = 6;"},

    {"question": "What is the average wind speed in the weather data?",
     "sql": "SELECT AVG(wind_speed_mps) FROM weather;"},

    {"question": "How many weather records have humidity above 80 percent?",
     "sql": "SELECT COUNT(*) FROM weather WHERE rel_humidity > 80;"},

    {"question": "How many trips were taken by annual pass holders?",
     "sql": "SELECT COUNT(*) FROM trips WHERE passholder_type = 'Annual Pass';"},

    {"question": "What is the average duration of electric bike trips?",
     "sql": "SELECT AVG(duration_minutes) FROM trips WHERE bike_type = 'Electric';"},

    {"question": "What is the average duration of standard bike trips?",
     "sql": "SELECT AVG(duration_minutes) FROM trips WHERE bike_type = 'Standard';"},

    {"question": "How many trips were taken in the afternoon between 12pm and 5pm?",
     "sql": "SELECT COUNT(*) FROM trips WHERE HOUR(start_time) BETWEEN 12 AND 16;"},

    {"question": "How many trips lasted between 30 and 60 minutes?",
     "sql": "SELECT COUNT(*) FROM trips WHERE duration_minutes BETWEEN 30 AND 60;"},

    {"question": "How many unique bikes were used?",
     "sql": "SELECT COUNT(DISTINCT bike_id) FROM trips;"},

    {"question": "How many trips were taken in the first quarter?",
     "sql": "SELECT COUNT(*) FROM trips WHERE MONTH(start_time) IN (1, 2, 3);"},

    {"question": "How many trips were taken in the second quarter?",
     "sql": "SELECT COUNT(*) FROM trips WHERE MONTH(start_time) IN (4, 5, 6);"},

    {"question": "How many trips were taken in the third quarter?",
     "sql": "SELECT COUNT(*) FROM trips WHERE MONTH(start_time) IN (7, 8, 9);"},

    {"question": "How many trips were taken in the fourth quarter?",
     "sql": "SELECT COUNT(*) FROM trips WHERE MONTH(start_time) IN (10, 11, 12);"},

    {"question": "What is the average humidity in the weather data?",
     "sql": "SELECT AVG(rel_humidity) FROM weather;"},

    {"question": "What is the average pressure in the weather data?",
     "sql": "SELECT AVG(pressure_hpa) FROM weather;"},

    {"question": "How many trips lasted exactly 30 minutes?",
     "sql": "SELECT COUNT(*) FROM trips WHERE duration_minutes = 30;"},

    {"question": "How many trips were taken in spring?",
     "sql": "SELECT COUNT(*) FROM trips WHERE MONTH(start_time) IN (3, 4, 5);"},

    {"question": "How many trips were taken in fall?",
     "sql": "SELECT COUNT(*) FROM trips WHERE MONTH(start_time) IN (9, 10, 11);"},

    {"question": "What is the most common passholder type?",
     "sql": "SELECT passholder_type, COUNT(*) as count FROM trips GROUP BY passholder_type ORDER BY count DESC LIMIT 1;"},

    {"question": "What is the most common bike type?",
     "sql": "SELECT bike_type, COUNT(*) as count FROM trips GROUP BY bike_type ORDER BY count DESC LIMIT 1;"},

    {"question": "How many trips were taken each month?",
     "sql": "SELECT MONTH(start_time) as month, COUNT(*) as trips FROM trips GROUP BY MONTH(start_time) ORDER BY month;"},

    {"question": "How many trips were taken each day of the week?",
     "sql": "SELECT DAYOFWEEK(start_time) as day, COUNT(*) as trips FROM trips GROUP BY DAYOFWEEK(start_time) ORDER BY day;"},

    {"question": "How many trips were taken each hour of the day?",
     "sql": "SELECT HOUR(start_time) as hour, COUNT(*) as trips FROM trips GROUP BY HOUR(start_time) ORDER BY hour;"},

    {"question": "What is the average trip duration by bike type?",
     "sql": "SELECT bike_type, AVG(duration_minutes) as avg_duration FROM trips GROUP BY bike_type;"},

    {"question": "What is the average trip duration by passholder type?",
     "sql": "SELECT passholder_type, AVG(duration_minutes) as avg_duration FROM trips GROUP BY passholder_type;"},

    {"question": "How many trips were taken by each passholder type?",
     "sql": "SELECT passholder_type, COUNT(*) as trips FROM trips GROUP BY passholder_type ORDER BY trips DESC;"},

    {"question": "What is the total number of trips by bike type?",
     "sql": "SELECT bike_type, COUNT(*) as trips FROM trips GROUP BY bike_type ORDER BY trips DESC;"},

    {"question": "How many trips were taken on hot days when temperature exceeded 30 degrees?",
     "sql": "SELECT COUNT(*) FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp) WHERE w.temp_c > 30;"},

    {"question": "How many trips were taken on cold days when temperature was below 10 degrees?",
     "sql": "SELECT COUNT(*) FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp) WHERE w.temp_c < 10;"},

    {"question": "List all station names.",
     "sql": "SELECT station_name FROM stations;"},

    {"question": "List all station names and their coordinates.",
     "sql": "SELECT station_name, lat, lon FROM stations;"},

    {"question": "How many trips started at each station?",
     "sql": "SELECT start_station_id, COUNT(*) as trips FROM trips GROUP BY start_station_id ORDER BY trips DESC;"},

    {"question": "How many trips ended at each station?",
     "sql": "SELECT end_station_id, COUNT(*) as trips FROM trips GROUP BY end_station_id ORDER BY trips DESC;"},

    {"question": "What is the longest trip duration?",
     "sql": "SELECT MAX(duration_minutes) FROM trips;"},

    {"question": "What is the shortest trip duration?",
     "sql": "SELECT MIN(duration_minutes) FROM trips;"},

    {"question": "How many trips were taken on Saturdays?",
     "sql": "SELECT COUNT(*) FROM trips WHERE DAYOFWEEK(start_time) = 7;"},

    {"question": "How many trips were taken on Sundays?",
     "sql": "SELECT COUNT(*) FROM trips WHERE DAYOFWEEK(start_time) = 1;"},

    {"question": "How many trips were taken on Tuesdays?",
     "sql": "SELECT COUNT(*) FROM trips WHERE DAYOFWEEK(start_time) = 3;"},

    {"question": "How many trips were taken on Wednesdays?",
     "sql": "SELECT COUNT(*) FROM trips WHERE DAYOFWEEK(start_time) = 4;"},

    {"question": "How many trips were taken on Thursdays?",
     "sql": "SELECT COUNT(*) FROM trips WHERE DAYOFWEEK(start_time) = 5;"},

    {"question": "How many weather records are there in total?",
     "sql": "SELECT COUNT(*) FROM weather;"},

    {"question": "What is the maximum wind speed recorded?",
     "sql": "SELECT MAX(wind_speed_mps) FROM weather;"},

    {"question": "What is the maximum precipitation recorded?",
     "sql": "SELECT MAX(precip_mm) FROM weather;"},

    {"question": "How many trips were taken after 8pm?",
     "sql": "SELECT COUNT(*) FROM trips WHERE HOUR(start_time) >= 20;"},

    {"question": "How many trips were taken before 6am?",
     "sql": "SELECT COUNT(*) FROM trips WHERE HOUR(start_time) < 6;"},

    {"question": "What is the average trip duration on weekends?",
     "sql": "SELECT AVG(duration_minutes) FROM trips WHERE DAYOFWEEK(start_time) IN (1, 7);"},

    {"question": "What is the average trip duration on weekdays?",
     "sql": "SELECT AVG(duration_minutes) FROM trips WHERE DAYOFWEEK(start_time) BETWEEN 2 AND 6;"},

    {"question": "How many trips were taken in each year?",
     "sql": "SELECT YEAR(start_time) as year, COUNT(*) as trips FROM trips GROUP BY YEAR(start_time);"},

    {"question": "What is the average temperature in summer months?",
     "sql": "SELECT AVG(temp_c) FROM weather WHERE MONTH(timestamp) IN (6, 7, 8);"},

    {"question": "What is the average temperature in winter months?",
     "sql": "SELECT AVG(temp_c) FROM weather WHERE MONTH(timestamp) IN (12, 1, 2);"},

    {"question": "How many trips were taken during peak morning hours 7am to 9am?",
     "sql": "SELECT COUNT(*) FROM trips WHERE HOUR(start_time) BETWEEN 7 AND 9;"},

    {"question": "How many trips were taken during peak evening hours 5pm to 7pm?",
     "sql": "SELECT COUNT(*) FROM trips WHERE HOUR(start_time) BETWEEN 17 AND 19;"},

    {"question": "How many unique start stations are there?",
     "sql": "SELECT COUNT(DISTINCT start_station_id) FROM trips;"},

    {"question": "How many unique end stations are there?",
     "sql": "SELECT COUNT(DISTINCT end_station_id) FROM trips;"},

    {"question": "How many trips were taken on windy days with wind speed above 5 meters per second?",
     "sql": "SELECT COUNT(*) FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp) WHERE w.wind_speed_mps > 5;"},

    {"question": "How many trips had duration greater than the average duration?",
     "sql": "SELECT COUNT(*) FROM trips WHERE duration_minutes > (SELECT AVG(duration_minutes) FROM trips);"},

    {"question": "How many trips were taken per bike type per month?",
     "sql": "SELECT bike_type, MONTH(start_time) as month, COUNT(*) as trips FROM trips GROUP BY bike_type, MONTH(start_time) ORDER BY bike_type, month;"},

    {"question": "How many rainy hours were recorded in the weather data?",
     "sql": "SELECT COUNT(*) FROM weather WHERE precip_mm > 0;"},

    {"question": "What is the average trip duration for monthly pass holders?",
     "sql": "SELECT AVG(duration_minutes) FROM trips WHERE passholder_type = 'Monthly Pass';"},

    {"question": "How many trips were taken on days with no rain?",
     "sql": "SELECT COUNT(*) FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp) WHERE w.precip_mm = 0;"},

    {"question": "What is the total number of electric bike trips in summer?",
     "sql": "SELECT COUNT(*) FROM trips WHERE bike_type = 'Electric' AND MONTH(start_time) IN (6, 7, 8);"},

    {"question": "How many trips were taken on the first day of each month?",
     "sql": "SELECT COUNT(*) FROM trips WHERE DAY(start_time) = 1;"},

    {"question": "What is the average trip duration in the morning peak hours?",
     "sql": "SELECT AVG(duration_minutes) FROM trips WHERE HOUR(start_time) BETWEEN 7 AND 9;"},

    {"question": "How many trips were taken on warm days above 25 degrees?",
     "sql": "SELECT COUNT(*) FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp) WHERE w.temp_c > 25;"},

    {"question": "What is the total number of walk-up trips on weekends?",
     "sql": "SELECT COUNT(*) FROM trips WHERE passholder_type = 'Walk-up' AND DAYOFWEEK(start_time) IN (1, 7);"},

    # ========== MEDIUM (100 pairs) ==========
    # JOINs, GROUP BY, ORDER BY, HAVING

    {"question": "Which station had the most trips started from it?",
     "sql": """SELECT s.station_name, COUNT(*) as trip_count
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY trip_count DESC LIMIT 1;"""},

    {"question": "Which station had the most trips ended at it?",
     "sql": """SELECT s.station_name, COUNT(*) as trip_count
FROM trips t JOIN stations s ON t.end_station_id = s.station_id
GROUP BY s.station_name ORDER BY trip_count DESC LIMIT 1;"""},

    {"question": "What are the top 5 busiest start stations?",
     "sql": """SELECT s.station_name, COUNT(*) as trip_count
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY trip_count DESC LIMIT 5;"""},

    {"question": "What are the top 5 busiest end stations?",
     "sql": """SELECT s.station_name, COUNT(*) as trip_count
FROM trips t JOIN stations s ON t.end_station_id = s.station_id
GROUP BY s.station_name ORDER BY trip_count DESC LIMIT 5;"""},

    {"question": "What is the average trip duration per station?",
     "sql": """SELECT s.station_name, AVG(t.duration_minutes) as avg_duration
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY avg_duration DESC;"""},

    {"question": "Which stations have more than 1000 trips?",
     "sql": """SELECT s.station_name, COUNT(*) as trip_count
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name HAVING trip_count > 1000 ORDER BY trip_count DESC;"""},

    {"question": "Which stations have more than 500 electric bike trips?",
     "sql": """SELECT s.station_name, COUNT(*) as trip_count
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
WHERE t.bike_type = 'Electric'
GROUP BY s.station_name HAVING trip_count > 500 ORDER BY trip_count DESC;"""},

    {"question": "What is the average trip duration on rainy days vs non rainy days?",
     "sql": """SELECT
  CASE WHEN w.precip_mm > 0 THEN 'Rainy' ELSE 'Non-Rainy' END as weather_type,
  AVG(t.duration_minutes) as avg_duration
FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
GROUP BY weather_type;"""},

    {"question": "How many trips were taken on rainy days vs non rainy days?",
     "sql": """SELECT
  CASE WHEN w.precip_mm > 0 THEN 'Rainy' ELSE 'Non-Rainy' END as weather_type,
  COUNT(*) as trip_count
FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
GROUP BY weather_type;"""},

    {"question": "Which hour of the day has the most trips?",
     "sql": """SELECT HOUR(start_time) as hour, COUNT(*) as trip_count
FROM trips GROUP BY hour ORDER BY trip_count DESC LIMIT 1;"""},

    {"question": "Which day of the week has the most trips?",
     "sql": """SELECT DAYOFWEEK(start_time) as day, COUNT(*) as trip_count
FROM trips GROUP BY day ORDER BY trip_count DESC LIMIT 1;"""},

    {"question": "Which month has the most trips?",
     "sql": """SELECT MONTH(start_time) as month, COUNT(*) as trip_count
FROM trips GROUP BY month ORDER BY trip_count DESC LIMIT 1;"""},

    {"question": "What is the number of trips per station per month?",
     "sql": """SELECT s.station_name, MONTH(t.start_time) as month, COUNT(*) as trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name, month ORDER BY s.station_name, month;"""},

    {"question": "Which station has the longest average trip duration?",
     "sql": """SELECT s.station_name, AVG(t.duration_minutes) as avg_duration
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY avg_duration DESC LIMIT 1;"""},

    {"question": "Which station has the shortest average trip duration?",
     "sql": """SELECT s.station_name, AVG(t.duration_minutes) as avg_duration
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY avg_duration ASC LIMIT 1;"""},

    {"question": "How many trips per month for electric bikes?",
     "sql": """SELECT MONTH(start_time) as month, COUNT(*) as trips
FROM trips WHERE bike_type = 'Electric'
GROUP BY month ORDER BY month;"""},

    {"question": "How many trips per month for standard bikes?",
     "sql": """SELECT MONTH(start_time) as month, COUNT(*) as trips
FROM trips WHERE bike_type = 'Standard'
GROUP BY month ORDER BY month;"""},

    {"question": "What is the busiest hour for electric bikes?",
     "sql": """SELECT HOUR(start_time) as hour, COUNT(*) as trips
FROM trips WHERE bike_type = 'Electric'
GROUP BY hour ORDER BY trips DESC LIMIT 1;"""},

    {"question": "Which passholder type takes the longest trips on average?",
     "sql": """SELECT passholder_type, AVG(duration_minutes) as avg_duration
FROM trips GROUP BY passholder_type ORDER BY avg_duration DESC LIMIT 1;"""},

    {"question": "What is the total trips and average duration per passholder type?",
     "sql": """SELECT passholder_type, COUNT(*) as total_trips, AVG(duration_minutes) as avg_duration
FROM trips GROUP BY passholder_type ORDER BY total_trips DESC;"""},

    {"question": "What are the top 10 most used bikes?",
     "sql": """SELECT bike_id, COUNT(*) as trips
FROM trips GROUP BY bike_id ORDER BY trips DESC LIMIT 10;"""},

    {"question": "What is the maximum number of trips taken in a single day?",
     "sql": """SELECT DATE(start_time) as trip_date, COUNT(*) as trips
FROM trips GROUP BY trip_date ORDER BY trips DESC LIMIT 1;"""},

    {"question": "How many trips were taken from each station on weekends?",
     "sql": """SELECT s.station_name, COUNT(*) as weekend_trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
WHERE DAYOFWEEK(t.start_time) IN (1, 7)
GROUP BY s.station_name ORDER BY weekend_trips DESC;"""},

    {"question": "How many trips were taken from each station on weekdays?",
     "sql": """SELECT s.station_name, COUNT(*) as weekday_trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
WHERE DAYOFWEEK(t.start_time) BETWEEN 2 AND 6
GROUP BY s.station_name ORDER BY weekday_trips DESC;"""},

    {"question": "What is the ratio of electric to standard bike trips per month?",
     "sql": """SELECT MONTH(start_time) as month,
  SUM(CASE WHEN bike_type = 'Electric' THEN 1 ELSE 0 END) as electric_trips,
  SUM(CASE WHEN bike_type = 'Standard' THEN 1 ELSE 0 END) as standard_trips
FROM trips GROUP BY month ORDER BY month;"""},

    {"question": "What is the average trip duration when temperature is above 25 degrees?",
     "sql": """SELECT AVG(t.duration_minutes) as avg_duration
FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
WHERE w.temp_c > 25;"""},

    {"question": "What is the average trip duration when temperature is below 15 degrees?",
     "sql": """SELECT AVG(t.duration_minutes) as avg_duration
FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
WHERE w.temp_c < 15;"""},

    {"question": "Which month has the highest average temperature?",
     "sql": """SELECT MONTH(timestamp) as month, AVG(temp_c) as avg_temp
FROM weather GROUP BY month ORDER BY avg_temp DESC LIMIT 1;"""},

    {"question": "Which month has the lowest average temperature?",
     "sql": """SELECT MONTH(timestamp) as month, AVG(temp_c) as avg_temp
FROM weather GROUP BY month ORDER BY avg_temp ASC LIMIT 1;"""},

    {"question": "What are the top 5 most popular routes between stations?",
     "sql": """SELECT start_station_id, end_station_id, COUNT(*) as trip_count
FROM trips GROUP BY start_station_id, end_station_id
ORDER BY trip_count DESC LIMIT 5;"""},

    {"question": "How many round trips were taken per station?",
     "sql": """SELECT s.station_name, COUNT(*) as round_trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
WHERE t.start_station_id = t.end_station_id
GROUP BY s.station_name ORDER BY round_trips DESC;"""},

    {"question": "How many trips were taken per hour on weekdays vs weekends?",
     "sql": """SELECT HOUR(start_time) as hour,
  SUM(CASE WHEN DAYOFWEEK(start_time) BETWEEN 2 AND 6 THEN 1 ELSE 0 END) as weekday_trips,
  SUM(CASE WHEN DAYOFWEEK(start_time) IN (1,7) THEN 1 ELSE 0 END) as weekend_trips
FROM trips GROUP BY hour ORDER BY hour;"""},

    {"question": "Which bikes have been used for more than 100 trips?",
     "sql": """SELECT bike_id, COUNT(*) as trips
FROM trips GROUP BY bike_id HAVING trips > 100 ORDER BY trips DESC;"""},

    {"question": "What is the total trip duration per station in hours?",
     "sql": """SELECT s.station_name, SUM(t.duration_minutes)/60 as total_hours
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY total_hours DESC;"""},

    {"question": "How many trips were taken per station per bike type?",
     "sql": """SELECT s.station_name, t.bike_type, COUNT(*) as trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name, t.bike_type ORDER BY s.station_name, trips DESC;"""},

    {"question": "How many trips were taken during high humidity above 70 percent?",
     "sql": """SELECT COUNT(*) as trips
FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
WHERE w.rel_humidity > 70;"""},

    {"question": "What is the trip count by passholder type and bike type?",
     "sql": """SELECT passholder_type, bike_type, COUNT(*) as trips
FROM trips GROUP BY passholder_type, bike_type ORDER BY passholder_type, trips DESC;"""},

    {"question": "Which stations have the highest electric bike usage ratio?",
     "sql": """SELECT s.station_name,
  SUM(CASE WHEN t.bike_type = 'Electric' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as electric_pct
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY electric_pct DESC LIMIT 10;"""},

    {"question": "How many trips were taken per season?",
     "sql": """SELECT
  CASE
    WHEN MONTH(start_time) IN (3,4,5) THEN 'Spring'
    WHEN MONTH(start_time) IN (6,7,8) THEN 'Summer'
    WHEN MONTH(start_time) IN (9,10,11) THEN 'Fall'
    ELSE 'Winter'
  END as season,
  COUNT(*) as trips
FROM trips GROUP BY season ORDER BY trips DESC;"""},

    {"question": "What is the daily average number of trips?",
     "sql": """SELECT AVG(daily_trips) as avg_daily_trips
FROM (SELECT DATE(start_time) as trip_date, COUNT(*) as daily_trips
      FROM trips GROUP BY trip_date) daily;"""},

    {"question": "Which stations are used more on weekends than weekdays?",
     "sql": """SELECT s.station_name,
  SUM(CASE WHEN DAYOFWEEK(t.start_time) IN (1,7) THEN 1 ELSE 0 END) as weekend_trips,
  SUM(CASE WHEN DAYOFWEEK(t.start_time) BETWEEN 2 AND 6 THEN 1 ELSE 0 END) as weekday_trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
HAVING weekend_trips > weekday_trips
ORDER BY weekend_trips DESC;"""},

    {"question": "What is the average number of daily trips per station?",
     "sql": """SELECT s.station_name, AVG(daily.daily_trips) as avg_daily
FROM (SELECT start_station_id, DATE(start_time) as trip_date, COUNT(*) as daily_trips
      FROM trips GROUP BY start_station_id, trip_date) daily
JOIN stations s ON daily.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY avg_daily DESC;"""},

    {"question": "How many trips were taken per weather condition code?",
     "sql": """SELECT w.weather_code, COUNT(*) as trips
FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
GROUP BY w.weather_code ORDER BY trips DESC;"""},

    {"question": "What is the average trip duration per weather condition?",
     "sql": """SELECT w.weather_code, AVG(t.duration_minutes) as avg_duration
FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
GROUP BY w.weather_code ORDER BY avg_duration DESC;"""},

    {"question": "What is the average trip duration per day of week?",
     "sql": """SELECT DAYOFWEEK(start_time) as day, AVG(duration_minutes) as avg_duration
FROM trips GROUP BY day ORDER BY day;"""},

    {"question": "What is the average trip duration per hour of day?",
     "sql": """SELECT HOUR(start_time) as hour, AVG(duration_minutes) as avg_duration
FROM trips GROUP BY hour ORDER BY hour;"""},

    {"question": "How many trips were taken on days with above average temperature?",
     "sql": """SELECT COUNT(*) FROM trips t
JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
WHERE w.temp_c > (SELECT AVG(temp_c) FROM weather);"""},

    {"question": "Which station has the most round trips?",
     "sql": """SELECT s.station_name, COUNT(*) as round_trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
WHERE t.start_station_id = t.end_station_id
GROUP BY s.station_name ORDER BY round_trips DESC LIMIT 1;"""},

    {"question": "What is the average trip duration on hot days above 30 degrees?",
     "sql": """SELECT AVG(t.duration_minutes) as avg_duration
FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
WHERE w.temp_c > 30;"""},

    {"question": "How many trips were taken in each month by electric bikes?",
     "sql": """SELECT MONTH(start_time) as month, COUNT(*) as trips
FROM trips WHERE bike_type = 'Electric'
GROUP BY month ORDER BY month;"""},

    {"question": "What is the minimum number of trips taken in a single day?",
     "sql": """SELECT DATE(start_time) as trip_date, COUNT(*) as trips
FROM trips GROUP BY trip_date ORDER BY trips ASC LIMIT 1;"""},

    {"question": "How many trips were taken at each hour on weekends?",
     "sql": """SELECT HOUR(start_time) as hour, COUNT(*) as trips
FROM trips WHERE DAYOFWEEK(start_time) IN (1,7)
GROUP BY hour ORDER BY hour;"""},

    {"question": "What is the average trip duration per month?",
     "sql": """SELECT MONTH(start_time) as month, AVG(duration_minutes) as avg_duration
FROM trips GROUP BY month ORDER BY month;"""},

    {"question": "Which station has the highest number of walk-up trips?",
     "sql": """SELECT s.station_name, COUNT(*) as trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
WHERE t.passholder_type = 'Walk-up'
GROUP BY s.station_name ORDER BY trips DESC LIMIT 1;"""},

    {"question": "How many trips were taken per passholder type per month?",
     "sql": """SELECT passholder_type, MONTH(start_time) as month, COUNT(*) as trips
FROM trips GROUP BY passholder_type, month ORDER BY passholder_type, month;"""},

    {"question": "What is the total trip duration per month in hours?",
     "sql": """SELECT MONTH(start_time) as month, SUM(duration_minutes)/60 as total_hours
FROM trips GROUP BY month ORDER BY month;"""},

    {"question": "Which day of the week has the longest average trip duration?",
     "sql": """SELECT DAYOFWEEK(start_time) as day, AVG(duration_minutes) as avg_duration
FROM trips GROUP BY day ORDER BY avg_duration DESC LIMIT 1;"""},

    {"question": "How many trips were taken per hour on rainy days?",
     "sql": """SELECT HOUR(t.start_time) as hour, COUNT(*) as trips
FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
WHERE w.precip_mm > 0
GROUP BY hour ORDER BY hour;"""},

    {"question": "What are the bottom 5 least used start stations?",
     "sql": """SELECT s.station_name, COUNT(*) as trip_count
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY trip_count ASC LIMIT 5;"""},

    {"question": "How many trips were taken per bike type on weekends?",
     "sql": """SELECT bike_type, COUNT(*) as trips
FROM trips WHERE DAYOFWEEK(start_time) IN (1,7)
GROUP BY bike_type ORDER BY trips DESC;"""},

    {"question": "How many trips were taken per bike type on weekdays?",
     "sql": """SELECT bike_type, COUNT(*) as trips
FROM trips WHERE DAYOFWEEK(start_time) BETWEEN 2 AND 6
GROUP BY bike_type ORDER BY trips DESC;"""},

    {"question": "What is the average wind speed on days with high ridership above 500 trips?",
     "sql": """SELECT AVG(w.wind_speed_mps) as avg_wind
FROM weather w
JOIN (SELECT DATE(start_time) as trip_date, COUNT(*) as daily_trips
      FROM trips GROUP BY trip_date HAVING daily_trips > 500) d
ON DATE(w.timestamp) = d.trip_date;"""},

    {"question": "Which passholder type has the most trips on weekends?",
     "sql": """SELECT passholder_type, COUNT(*) as trips
FROM trips WHERE DAYOFWEEK(start_time) IN (1,7)
GROUP BY passholder_type ORDER BY trips DESC LIMIT 1;"""},

    {"question": "How many trips were taken from the top 3 stations combined?",
     "sql": """SELECT SUM(trip_count) as total_trips
FROM (SELECT start_station_id, COUNT(*) as trip_count
      FROM trips GROUP BY start_station_id
      ORDER BY trip_count DESC LIMIT 3) top3;"""},

    {"question": "What is the average duration of trips that start and end at different stations?",
     "sql": """SELECT AVG(duration_minutes) as avg_duration
FROM trips WHERE start_station_id != end_station_id;"""},

    {"question": "How many trips were taken on days with high wind speed above 10 meters per second?",
     "sql": """SELECT COUNT(*) as trips
FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
WHERE w.wind_speed_mps > 10;"""},

    {"question": "Which station pair has the most trips between them?",
     "sql": """SELECT s1.station_name as start_station, s2.station_name as end_station,
COUNT(*) as trips
FROM trips t
JOIN stations s1 ON t.start_station_id = s1.station_id
JOIN stations s2 ON t.end_station_id = s2.station_id
WHERE t.start_station_id != t.end_station_id
GROUP BY s1.station_name, s2.station_name
ORDER BY trips DESC LIMIT 1;"""},

    {"question": "What is the average trip duration per season?",
     "sql": """SELECT
  CASE
    WHEN MONTH(start_time) IN (3,4,5) THEN 'Spring'
    WHEN MONTH(start_time) IN (6,7,8) THEN 'Summer'
    WHEN MONTH(start_time) IN (9,10,11) THEN 'Fall'
    ELSE 'Winter'
  END as season,
  AVG(duration_minutes) as avg_duration
FROM trips GROUP BY season;"""},

    {"question": "How many trips were taken per hour during summer months?",
     "sql": """SELECT HOUR(start_time) as hour, COUNT(*) as trips
FROM trips WHERE MONTH(start_time) IN (6,7,8)
GROUP BY hour ORDER BY hour;"""},

    {"question": "What is the average temperature per season?",
     "sql": """SELECT
  CASE
    WHEN MONTH(timestamp) IN (3,4,5) THEN 'Spring'
    WHEN MONTH(timestamp) IN (6,7,8) THEN 'Summer'
    WHEN MONTH(timestamp) IN (9,10,11) THEN 'Fall'
    ELSE 'Winter'
  END as season,
  AVG(temp_c) as avg_temp
FROM weather GROUP BY season;"""},

    {"question": "How many trips per month had duration above 45 minutes?",
     "sql": """SELECT MONTH(start_time) as month, COUNT(*) as trips
FROM trips WHERE duration_minutes > 45
GROUP BY month ORDER BY month;"""},

    {"question": "Which hour has the highest average trip duration on weekends?",
     "sql": """SELECT HOUR(start_time) as hour, AVG(duration_minutes) as avg_duration
FROM trips WHERE DAYOFWEEK(start_time) IN (1,7)
GROUP BY hour ORDER BY avg_duration DESC LIMIT 1;"""},

    {"question": "How many trips were taken on days with precipitation above 5mm?",
     "sql": """SELECT COUNT(*) as trips
FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
WHERE w.precip_mm > 5;"""},

    {"question": "What is the total number of trips per station in the top 10?",
     "sql": """SELECT s.station_name, COUNT(*) as trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY trips DESC LIMIT 10;"""},

    {"question": "How many trips started at each station by bike type?",
     "sql": """SELECT s.station_name, t.bike_type, COUNT(*) as trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name, t.bike_type
ORDER BY s.station_name, trips DESC;"""},

    {"question": "What is the average humidity on high ridership days above 1000 trips?",
     "sql": """SELECT AVG(w.rel_humidity) as avg_humidity
FROM weather w
JOIN (SELECT DATE(start_time) as trip_date, COUNT(*) as daily_trips
      FROM trips GROUP BY trip_date HAVING daily_trips > 1000) d
ON DATE(w.timestamp) = d.trip_date;"""},

    # ========== HARD (100 pairs) ==========
    # Subqueries, HAVING with subqueries, window functions, CASE WHEN, CTEs

    {"question": "Which stations have above average number of trips?",
     "sql": """SELECT s.station_name, COUNT(*) as trip_count
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
HAVING trip_count > (
    SELECT AVG(cnt) FROM (
        SELECT COUNT(*) as cnt FROM trips GROUP BY start_station_id
    ) sub
)
ORDER BY trip_count DESC;"""},

    {"question": "Which stations have above average electric bike usage?",
     "sql": """SELECT s.station_name, COUNT(*) as electric_trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
WHERE t.bike_type = 'Electric'
GROUP BY s.station_name
HAVING electric_trips > (
    SELECT AVG(cnt) FROM (
        SELECT COUNT(*) as cnt FROM trips
        WHERE bike_type = 'Electric'
        GROUP BY start_station_id
    ) sub
)
ORDER BY electric_trips DESC;"""},

    {"question": "Find stations where weekend trips exceed weekday trips by more than 20 percent?",
     "sql": """SELECT s.station_name,
  SUM(CASE WHEN DAYOFWEEK(t.start_time) IN (1,7) THEN 1 ELSE 0 END) as weekend_trips,
  SUM(CASE WHEN DAYOFWEEK(t.start_time) BETWEEN 2 AND 6 THEN 1 ELSE 0 END) as weekday_trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
HAVING weekend_trips > weekday_trips * 1.2
ORDER BY weekend_trips DESC;"""},

    {"question": "Which bikes have been used in more than 3 different stations?",
     "sql": """SELECT bike_id, COUNT(DISTINCT start_station_id) as station_count
FROM trips
GROUP BY bike_id
HAVING station_count > 3
ORDER BY station_count DESC;"""},

    {"question": "Find the month with the highest ratio of electric to standard bike trips?",
     "sql": """SELECT month,
  electric_trips * 1.0 / NULLIF(standard_trips, 0) as ratio
FROM (
  SELECT MONTH(start_time) as month,
    SUM(CASE WHEN bike_type = 'Electric' THEN 1 ELSE 0 END) as electric_trips,
    SUM(CASE WHEN bike_type = 'Standard' THEN 1 ELSE 0 END) as standard_trips
  FROM trips GROUP BY MONTH(start_time)
) monthly
ORDER BY ratio DESC LIMIT 1;"""},

    {"question": "Which stations had more trips in the second half of the year than the first half?",
     "sql": """SELECT s.station_name,
  SUM(CASE WHEN MONTH(t.start_time) <= 6 THEN 1 ELSE 0 END) as first_half,
  SUM(CASE WHEN MONTH(t.start_time) > 6 THEN 1 ELSE 0 END) as second_half
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
HAVING second_half > first_half
ORDER BY second_half DESC;"""},

    {"question": "What is the running total of trips by month?",
     "sql": """SELECT month, monthly_trips,
  SUM(monthly_trips) OVER (ORDER BY month) as running_total
FROM (
  SELECT MONTH(start_time) as month, COUNT(*) as monthly_trips
  FROM trips GROUP BY MONTH(start_time)
) monthly
ORDER BY month;"""},

    {"question": "Rank stations by total trips using dense rank?",
     "sql": """SELECT s.station_name, COUNT(*) as trips,
  DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) as rank
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY rank;"""},

    {"question": "What is the percentage of total trips each station contributes?",
     "sql": """SELECT s.station_name,
  COUNT(*) as trips,
  COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () as pct_of_total
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY trips DESC;"""},

    {"question": "Which stations have never had a round trip?",
     "sql": """SELECT s.station_name
FROM stations s
WHERE s.station_id NOT IN (
    SELECT DISTINCT start_station_id FROM trips
    WHERE start_station_id = end_station_id
)
ORDER BY s.station_name;"""},

    {"question": "Find stations that appear in both top 10 start and top 10 end stations?",
     "sql": """SELECT station_name FROM (
  SELECT s.station_name FROM trips t
  JOIN stations s ON t.start_station_id = s.station_id
  GROUP BY s.station_name ORDER BY COUNT(*) DESC LIMIT 10
) top_start
INTERSECT
SELECT station_name FROM (
  SELECT s.station_name FROM trips t
  JOIN stations s ON t.end_station_id = s.station_id
  GROUP BY s.station_name ORDER BY COUNT(*) DESC LIMIT 10
) top_end;"""},

    {"question": "What is the month over month growth rate of trips?",
     "sql": """SELECT month,
  monthly_trips,
  LAG(monthly_trips) OVER (ORDER BY month) as prev_month,
  (monthly_trips - LAG(monthly_trips) OVER (ORDER BY month)) * 100.0 /
    NULLIF(LAG(monthly_trips) OVER (ORDER BY month), 0) as growth_pct
FROM (
  SELECT MONTH(start_time) as month, COUNT(*) as monthly_trips
  FROM trips GROUP BY MONTH(start_time)
) monthly
ORDER BY month;"""},

    {"question": "Which bikes have the highest average trip duration above the overall average?",
     "sql": """SELECT bike_id, AVG(duration_minutes) as avg_duration
FROM trips
GROUP BY bike_id
HAVING AVG(duration_minutes) > (SELECT AVG(duration_minutes) FROM trips)
ORDER BY avg_duration DESC LIMIT 10;"""},

    {"question": "Find all stations that have trips in every month of the year?",
     "sql": """SELECT start_station_id
FROM trips
GROUP BY start_station_id
HAVING COUNT(DISTINCT MONTH(start_time)) = 12;"""},

    {"question": "What is the average trip duration for each station compared to the overall average?",
     "sql": """SELECT s.station_name,
  AVG(t.duration_minutes) as station_avg,
  (SELECT AVG(duration_minutes) FROM trips) as overall_avg,
  AVG(t.duration_minutes) - (SELECT AVG(duration_minutes) FROM trips) as difference
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY difference DESC;"""},

    {"question": "Which day had the most diverse usage across bike types?",
     "sql": """SELECT DATE(start_time) as trip_date,
  COUNT(DISTINCT bike_type) as bike_type_count,
  COUNT(*) as total_trips
FROM trips
GROUP BY trip_date
ORDER BY bike_type_count DESC, total_trips DESC
LIMIT 1;"""},

    {"question": "Find the top 3 stations for each bike type?",
     "sql": """SELECT bike_type, station_name, trips FROM (
  SELECT t.bike_type, s.station_name, COUNT(*) as trips,
    ROW_NUMBER() OVER (PARTITION BY t.bike_type ORDER BY COUNT(*) DESC) as rn
  FROM trips t JOIN stations s ON t.start_station_id = s.station_id
  GROUP BY t.bike_type, s.station_name
) ranked
WHERE rn <= 3
ORDER BY bike_type, trips DESC;"""},

    {"question": "What is the cumulative percentage of trips by station ordered by popularity?",
     "sql": """SELECT station_name, trips,
  SUM(trips) OVER (ORDER BY trips DESC) * 100.0 /
    SUM(trips) OVER () as cumulative_pct
FROM (
  SELECT s.station_name, COUNT(*) as trips
  FROM trips t JOIN stations s ON t.start_station_id = s.station_id
  GROUP BY s.station_name
) station_trips
ORDER BY trips DESC;"""},

    {"question": "Which hours have above median trip counts?",
     "sql": """SELECT hour, hourly_trips FROM (
  SELECT HOUR(start_time) as hour, COUNT(*) as hourly_trips
  FROM trips GROUP BY HOUR(start_time)
) hourly
WHERE hourly_trips > (
  SELECT AVG(hourly_trips) FROM (
    SELECT COUNT(*) as hourly_trips
    FROM trips GROUP BY HOUR(start_time)
  ) sub
)
ORDER BY hourly_trips DESC;"""},

    {"question": "Find stations where the average trip duration increased from first half to second half of the year?",
     "sql": """SELECT s.station_name,
  AVG(CASE WHEN MONTH(t.start_time) <= 6 THEN t.duration_minutes END) as h1_avg,
  AVG(CASE WHEN MONTH(t.start_time) > 6 THEN t.duration_minutes END) as h2_avg
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
HAVING h2_avg > h1_avg
ORDER BY (h2_avg - h1_avg) DESC;"""},

    {"question": "What is the correlation between temperature and number of trips per day?",
     "sql": """SELECT
  AVG(w.temp_c) as avg_temp,
  COUNT(t.trip_id) as daily_trips
FROM trips t
JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
GROUP BY DATE(t.start_time)
ORDER BY avg_temp;"""},

    {"question": "Which stations have consistent ridership across all seasons?",
     "sql": """SELECT s.station_name,
  MAX(seasonal_trips) - MIN(seasonal_trips) as trip_variance
FROM (
  SELECT t.start_station_id,
    CASE
      WHEN MONTH(t.start_time) IN (3,4,5) THEN 'Spring'
      WHEN MONTH(t.start_time) IN (6,7,8) THEN 'Summer'
      WHEN MONTH(t.start_time) IN (9,10,11) THEN 'Fall'
      ELSE 'Winter'
    END as season,
    COUNT(*) as seasonal_trips
  FROM trips t
  GROUP BY t.start_station_id, season
) seasonal
JOIN stations s ON seasonal.start_station_id = s.station_id
GROUP BY s.station_name
HAVING COUNT(DISTINCT season) = 4
ORDER BY trip_variance ASC
LIMIT 10;"""},

    {"question": "Find the most popular trip route for each hour of the day?",
     "sql": """SELECT hour, start_station_id, end_station_id, trips FROM (
  SELECT HOUR(start_time) as hour,
    start_station_id, end_station_id,
    COUNT(*) as trips,
    ROW_NUMBER() OVER (PARTITION BY HOUR(start_time) ORDER BY COUNT(*) DESC) as rn
  FROM trips
  WHERE start_station_id != end_station_id
  GROUP BY HOUR(start_time), start_station_id, end_station_id
) ranked
WHERE rn = 1
ORDER BY hour;"""},

    {"question": "Which passholder type has the most consistent trip duration across months?",
     "sql": """SELECT passholder_type,
  STDDEV(monthly_avg) as duration_stddev
FROM (
  SELECT passholder_type,
    MONTH(start_time) as month,
    AVG(duration_minutes) as monthly_avg
  FROM trips
  GROUP BY passholder_type, MONTH(start_time)
) monthly_avgs
GROUP BY passholder_type
ORDER BY duration_stddev ASC;"""},

    {"question": "What is the busiest 3 hour window of the day system wide?",
     "sql": """SELECT start_hour,
  SUM(hourly_trips) OVER (ORDER BY start_hour ROWS BETWEEN CURRENT ROW AND 2 FOLLOWING) as window_trips
FROM (
  SELECT HOUR(start_time) as start_hour, COUNT(*) as hourly_trips
  FROM trips GROUP BY HOUR(start_time)
) hourly
ORDER BY window_trips DESC
LIMIT 1;"""},

    {"question": "Which stations have a higher proportion of long trips above 45 minutes than the system average?",
     "sql": """SELECT s.station_name,
  SUM(CASE WHEN t.duration_minutes > 45 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as long_trip_pct
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
HAVING long_trip_pct > (
  SELECT SUM(CASE WHEN duration_minutes > 45 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
  FROM trips
)
ORDER BY long_trip_pct DESC;"""},

    {"question": "Find months where electric bike trips exceeded standard bike trips?",
     "sql": """SELECT month FROM (
  SELECT MONTH(start_time) as month,
    SUM(CASE WHEN bike_type = 'Electric' THEN 1 ELSE 0 END) as electric,
    SUM(CASE WHEN bike_type = 'Standard' THEN 1 ELSE 0 END) as standard
  FROM trips GROUP BY MONTH(start_time)
) monthly
WHERE electric > standard
ORDER BY month;"""},

    {"question": "What is the average trip duration percentile for each station?",
     "sql": """SELECT s.station_name,
  AVG(t.duration_minutes) as avg_duration,
  PERCENT_RANK() OVER (ORDER BY AVG(t.duration_minutes)) as duration_percentile
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY duration_percentile DESC;"""},

    {"question": "Which stations had their peak ridership month in summer?",
     "sql": """SELECT s.station_name, peak_month
FROM (
  SELECT start_station_id,
    MONTH(start_time) as peak_month,
    COUNT(*) as trips,
    ROW_NUMBER() OVER (PARTITION BY start_station_id ORDER BY COUNT(*) DESC) as rn
  FROM trips GROUP BY start_station_id, MONTH(start_time)
) peak
JOIN stations s ON peak.start_station_id = s.station_id
WHERE rn = 1 AND peak_month IN (6, 7, 8)
ORDER BY s.station_name;"""},

    {"question": "Find the top 5 stations with the most growth between first and second half of the year?",
     "sql": """SELECT s.station_name,
  first_half, second_half,
  second_half - first_half as growth
FROM (
  SELECT start_station_id,
    SUM(CASE WHEN MONTH(start_time) <= 6 THEN 1 ELSE 0 END) as first_half,
    SUM(CASE WHEN MONTH(start_time) > 6 THEN 1 ELSE 0 END) as second_half
  FROM trips GROUP BY start_station_id
) halves
JOIN stations s ON halves.start_station_id = s.station_id
ORDER BY growth DESC LIMIT 5;"""},

    {"question": "What percentage of trips occur during peak hours 7 to 9 and 17 to 19 per station?",
     "sql": """SELECT s.station_name,
  SUM(CASE WHEN HOUR(t.start_time) BETWEEN 7 AND 9
           OR HOUR(t.start_time) BETWEEN 17 AND 19 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as peak_pct
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY peak_pct DESC;"""},

    {"question": "Which bike has the most trips across the most different stations?",
     "sql": """SELECT bike_id,
  COUNT(*) as total_trips,
  COUNT(DISTINCT start_station_id) as unique_stations
FROM trips
GROUP BY bike_id
ORDER BY unique_stations DESC, total_trips DESC
LIMIT 1;"""},

    {"question": "What is the average gap in minutes between consecutive trips at each station?",
     "sql": """SELECT start_station_id,
  AVG(gap_minutes) as avg_gap
FROM (
  SELECT start_station_id,
    TIMESTAMPDIFF(MINUTE,
      LAG(start_time) OVER (PARTITION BY start_station_id ORDER BY start_time),
      start_time) as gap_minutes
  FROM trips
) gaps
WHERE gap_minutes IS NOT NULL AND gap_minutes > 0
GROUP BY start_station_id
ORDER BY avg_gap ASC;"""},

    {"question": "Which stations have never been an end station?",
     "sql": """SELECT s.station_name
FROM stations s
WHERE s.station_id NOT IN (
    SELECT DISTINCT end_station_id FROM trips
    WHERE end_station_id IS NOT NULL
)
ORDER BY s.station_name;"""},

    {"question": "Find the hour with the highest standard deviation in trip counts across all days?",
     "sql": """SELECT hour, STDDEV(daily_count) as count_stddev FROM (
  SELECT HOUR(start_time) as hour,
    DATE(start_time) as trip_date,
    COUNT(*) as daily_count
  FROM trips
  GROUP BY HOUR(start_time), DATE(start_time)
) daily_hourly
GROUP BY hour
ORDER BY count_stddev DESC LIMIT 1;"""},

    {"question": "What is the average trip duration for trips that start at top 5 stations?",
     "sql": """SELECT AVG(duration_minutes) as avg_duration
FROM trips
WHERE start_station_id IN (
  SELECT start_station_id FROM trips
  GROUP BY start_station_id
  ORDER BY COUNT(*) DESC LIMIT 5
);"""},

    {"question": "Which station transitions from low to high ridership the fastest in the morning?",
     "sql": """SELECT start_station_id,
  MAX(CASE WHEN HOUR(start_time) = 9 THEN hourly_count ELSE 0 END) -
  MAX(CASE WHEN HOUR(start_time) = 6 THEN hourly_count ELSE 0 END) as morning_surge
FROM (
  SELECT start_station_id, HOUR(start_time) as hour,
    COUNT(*) as hourly_count
  FROM trips
  GROUP BY start_station_id, HOUR(start_time)
) hourly
GROUP BY start_station_id
ORDER BY morning_surge DESC LIMIT 1;"""},

    {"question": "Find stations where the most popular bike type changes between weekdays and weekends?",
     "sql": """SELECT wd.start_station_id
FROM (
  SELECT start_station_id,
    bike_type,
    ROW_NUMBER() OVER (PARTITION BY start_station_id ORDER BY COUNT(*) DESC) as rn
  FROM trips WHERE DAYOFWEEK(start_time) BETWEEN 2 AND 6
  GROUP BY start_station_id, bike_type
) wd
JOIN (
  SELECT start_station_id,
    bike_type,
    ROW_NUMBER() OVER (PARTITION BY start_station_id ORDER BY COUNT(*) DESC) as rn
  FROM trips WHERE DAYOFWEEK(start_time) IN (1,7)
  GROUP BY start_station_id, bike_type
) we ON wd.start_station_id = we.start_station_id
WHERE wd.rn = 1 AND we.rn = 1 AND wd.bike_type != we.bike_type;"""},

    {"question": "What is the trip count for each station normalized by the maximum station count?",
     "sql": """SELECT s.station_name,
  COUNT(*) as trips,
  COUNT(*) * 1.0 / MAX(COUNT(*)) OVER () as normalized_count
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY normalized_count DESC;"""},

    {"question": "Which months had both above average trips and above average temperature?",
     "sql": """SELECT t.month
FROM (
  SELECT MONTH(start_time) as month, COUNT(*) as trips
  FROM trips GROUP BY MONTH(start_time)
  HAVING trips > (SELECT AVG(monthly_trips) FROM
    (SELECT COUNT(*) as monthly_trips FROM trips GROUP BY MONTH(start_time)) m)
) t
JOIN (
  SELECT MONTH(timestamp) as month, AVG(temp_c) as avg_temp
  FROM weather GROUP BY MONTH(timestamp)
  HAVING avg_temp > (SELECT AVG(temp_c) FROM weather)
) w ON t.month = w.month
ORDER BY t.month;"""},

    {"question": "What is the median trip duration per station?",
     "sql": """SELECT s.station_name,
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY t.duration_minutes) as median_duration
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY median_duration DESC;"""},

    {"question": "Find stations where ridership dropped by more than 30 percent in winter compared to summer?",
     "sql": """SELECT s.station_name,
  summer_trips, winter_trips,
  (summer_trips - winter_trips) * 100.0 / NULLIF(summer_trips, 0) as drop_pct
FROM (
  SELECT start_station_id,
    SUM(CASE WHEN MONTH(start_time) IN (6,7,8) THEN 1 ELSE 0 END) as summer_trips,
    SUM(CASE WHEN MONTH(start_time) IN (12,1,2) THEN 1 ELSE 0 END) as winter_trips
  FROM trips GROUP BY start_station_id
) seasonal
JOIN stations s ON seasonal.start_station_id = s.station_id
HAVING drop_pct > 30
ORDER BY drop_pct DESC;"""},

    {"question": "Which passholder type has the highest trip count on rainy days relative to sunny days?",
     "sql": """SELECT passholder_type,
  rainy_trips * 1.0 / NULLIF(sunny_trips, 0) as rainy_to_sunny_ratio
FROM (
  SELECT t.passholder_type,
    SUM(CASE WHEN w.precip_mm > 0 THEN 1 ELSE 0 END) as rainy_trips,
    SUM(CASE WHEN w.precip_mm = 0 THEN 1 ELSE 0 END) as sunny_trips
  FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
  GROUP BY t.passholder_type
) rain_stats
ORDER BY rainy_to_sunny_ratio DESC LIMIT 1;"""},

]

print(f"Total bike-share pairs created: {len(bikeshare_pairs)}")

# Count by difficulty
easy  = bikeshare_pairs[:100]
medium = bikeshare_pairs[100:200]
hard  = bikeshare_pairs[200:]

print(f"  Easy   : {len(easy)}")
print(f"  Medium : {len(medium)}")
print(f"  Hard   : {len(hard)}")

# Show 1 example from each level
print("\n--- EASY EXAMPLE ---")
print(f"Q: {easy[0]['question']}")
print(f"SQL: {easy[0]['sql']}")

print("\n--- MEDIUM EXAMPLE ---")
print(f"Q: {medium[0]['question']}")
print(f"SQL: {medium[0]['sql']}")

print("\n--- HARD EXAMPLE ---")
print(f"Q: {hard[0]['question']}")
print(f"SQL: {hard[0]['sql']}")

Total bike-share pairs created: 215
  Easy   : 100
  Medium : 100
  Hard   : 15

--- EASY EXAMPLE ---
Q: How many trips were taken in total?
SQL: SELECT COUNT(*) FROM trips;

--- MEDIUM EXAMPLE ---
Q: What is the average trip duration per station?
SQL: SELECT s.station_name, AVG(t.duration_minutes) as avg_duration
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY avg_duration DESC;

--- HARD EXAMPLE ---
Q: Which stations had their peak ridership month in summer?
SQL: SELECT s.station_name, peak_month
FROM (
  SELECT start_station_id,
    MONTH(start_time) as peak_month,
    COUNT(*) as trips,
    ROW_NUMBER() OVER (PARTITION BY start_station_id ORDER BY COUNT(*) DESC) as rn
  FROM trips GROUP BY start_station_id, MONTH(start_time)
) peak
JOIN stations s ON peak.start_station_id = s.station_id
WHERE rn = 1 AND peak_month IN (6, 7, 8)
ORDER BY s.station_name;


In [36]:
# Add remaining hard pairs
hard_pairs_remaining = [

    {"question": "What is the running total of trips by month?",
     "sql": """SELECT month, monthly_trips,
  SUM(monthly_trips) OVER (ORDER BY month) as running_total
FROM (
  SELECT MONTH(start_time) as month, COUNT(*) as monthly_trips
  FROM trips GROUP BY MONTH(start_time)
) monthly
ORDER BY month;"""},

    {"question": "Rank stations by total trips using dense rank?",
     "sql": """SELECT s.station_name, COUNT(*) as trips,
  DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) as rank
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY rank;"""},

    {"question": "What is the percentage of total trips each station contributes?",
     "sql": """SELECT s.station_name,
  COUNT(*) as trips,
  COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () as pct_of_total
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY trips DESC;"""},

    {"question": "Which stations have never had a round trip?",
     "sql": """SELECT s.station_name
FROM stations s
WHERE s.station_id NOT IN (
    SELECT DISTINCT start_station_id FROM trips
    WHERE start_station_id = end_station_id
)
ORDER BY s.station_name;"""},

    {"question": "Find stations that appear in both top 10 start and top 10 end stations?",
     "sql": """SELECT station_name FROM (
  SELECT s.station_name FROM trips t
  JOIN stations s ON t.start_station_id = s.station_id
  GROUP BY s.station_name ORDER BY COUNT(*) DESC LIMIT 10
) top_start
INTERSECT
SELECT station_name FROM (
  SELECT s.station_name FROM trips t
  JOIN stations s ON t.end_station_id = s.station_id
  GROUP BY s.station_name ORDER BY COUNT(*) DESC LIMIT 10
) top_end;"""},

    {"question": "What is the month over month growth rate of trips?",
     "sql": """SELECT month, monthly_trips,
  LAG(monthly_trips) OVER (ORDER BY month) as prev_month,
  (monthly_trips - LAG(monthly_trips) OVER (ORDER BY month)) * 100.0 /
    NULLIF(LAG(monthly_trips) OVER (ORDER BY month), 0) as growth_pct
FROM (
  SELECT MONTH(start_time) as month, COUNT(*) as monthly_trips
  FROM trips GROUP BY MONTH(start_time)
) monthly
ORDER BY month;"""},

    {"question": "Which bikes have the highest average trip duration above the overall average?",
     "sql": """SELECT bike_id, AVG(duration_minutes) as avg_duration
FROM trips
GROUP BY bike_id
HAVING AVG(duration_minutes) > (SELECT AVG(duration_minutes) FROM trips)
ORDER BY avg_duration DESC LIMIT 10;"""},

    {"question": "Find all stations that have trips in every month of the year?",
     "sql": """SELECT start_station_id
FROM trips
GROUP BY start_station_id
HAVING COUNT(DISTINCT MONTH(start_time)) = 12;"""},

    {"question": "What is the average trip duration for each station compared to the overall average?",
     "sql": """SELECT s.station_name,
  AVG(t.duration_minutes) as station_avg,
  (SELECT AVG(duration_minutes) FROM trips) as overall_avg,
  AVG(t.duration_minutes) - (SELECT AVG(duration_minutes) FROM trips) as difference
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY difference DESC;"""},

    {"question": "Which day had the most diverse usage across bike types?",
     "sql": """SELECT DATE(start_time) as trip_date,
  COUNT(DISTINCT bike_type) as bike_type_count,
  COUNT(*) as total_trips
FROM trips
GROUP BY trip_date
ORDER BY bike_type_count DESC, total_trips DESC
LIMIT 1;"""},

    {"question": "Find the top 3 stations for each bike type?",
     "sql": """SELECT bike_type, station_name, trips FROM (
  SELECT t.bike_type, s.station_name, COUNT(*) as trips,
    ROW_NUMBER() OVER (PARTITION BY t.bike_type ORDER BY COUNT(*) DESC) as rn
  FROM trips t JOIN stations s ON t.start_station_id = s.station_id
  GROUP BY t.bike_type, s.station_name
) ranked
WHERE rn <= 3
ORDER BY bike_type, trips DESC;"""},

    {"question": "What is the cumulative percentage of trips by station ordered by popularity?",
     "sql": """SELECT station_name, trips,
  SUM(trips) OVER (ORDER BY trips DESC) * 100.0 /
    SUM(trips) OVER () as cumulative_pct
FROM (
  SELECT s.station_name, COUNT(*) as trips
  FROM trips t JOIN stations s ON t.start_station_id = s.station_id
  GROUP BY s.station_name
) station_trips
ORDER BY trips DESC;"""},

    {"question": "Which hours have above median trip counts?",
     "sql": """SELECT hour, hourly_trips FROM (
  SELECT HOUR(start_time) as hour, COUNT(*) as hourly_trips
  FROM trips GROUP BY HOUR(start_time)
) hourly
WHERE hourly_trips > (
  SELECT AVG(hourly_trips) FROM (
    SELECT COUNT(*) as hourly_trips
    FROM trips GROUP BY HOUR(start_time)
  ) sub
)
ORDER BY hourly_trips DESC;"""},

    {"question": "Find stations where the average trip duration increased from first half to second half of the year?",
     "sql": """SELECT s.station_name,
  AVG(CASE WHEN MONTH(t.start_time) <= 6 THEN t.duration_minutes END) as h1_avg,
  AVG(CASE WHEN MONTH(t.start_time) > 6 THEN t.duration_minutes END) as h2_avg
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
HAVING h2_avg > h1_avg
ORDER BY (h2_avg - h1_avg) DESC;"""},

    {"question": "What is the correlation between temperature and number of trips per day?",
     "sql": """SELECT AVG(w.temp_c) as avg_temp,
  COUNT(t.trip_id) as daily_trips
FROM trips t
JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
GROUP BY DATE(t.start_time)
ORDER BY avg_temp;"""},

    {"question": "Which stations have consistent ridership across all seasons?",
     "sql": """SELECT s.station_name,
  MAX(seasonal_trips) - MIN(seasonal_trips) as trip_variance
FROM (
  SELECT t.start_station_id,
    CASE
      WHEN MONTH(t.start_time) IN (3,4,5) THEN 'Spring'
      WHEN MONTH(t.start_time) IN (6,7,8) THEN 'Summer'
      WHEN MONTH(t.start_time) IN (9,10,11) THEN 'Fall'
      ELSE 'Winter'
    END as season,
    COUNT(*) as seasonal_trips
  FROM trips t
  GROUP BY t.start_station_id, season
) seasonal
JOIN stations s ON seasonal.start_station_id = s.station_id
GROUP BY s.station_name
HAVING COUNT(DISTINCT season) = 4
ORDER BY trip_variance ASC LIMIT 10;"""},

    {"question": "Find the most popular trip route for each hour of the day?",
     "sql": """SELECT hour, start_station_id, end_station_id, trips FROM (
  SELECT HOUR(start_time) as hour,
    start_station_id, end_station_id,
    COUNT(*) as trips,
    ROW_NUMBER() OVER (
      PARTITION BY HOUR(start_time)
      ORDER BY COUNT(*) DESC) as rn
  FROM trips
  WHERE start_station_id != end_station_id
  GROUP BY HOUR(start_time), start_station_id, end_station_id
) ranked
WHERE rn = 1
ORDER BY hour;"""},

    {"question": "Which passholder type has the most consistent trip duration across months?",
     "sql": """SELECT passholder_type,
  STDDEV(monthly_avg) as duration_stddev
FROM (
  SELECT passholder_type,
    MONTH(start_time) as month,
    AVG(duration_minutes) as monthly_avg
  FROM trips
  GROUP BY passholder_type, MONTH(start_time)
) monthly_avgs
GROUP BY passholder_type
ORDER BY duration_stddev ASC;"""},

    {"question": "What is the busiest 3 hour window of the day system wide?",
     "sql": """SELECT start_hour,
  SUM(hourly_trips) OVER (
    ORDER BY start_hour
    ROWS BETWEEN CURRENT ROW AND 2 FOLLOWING) as window_trips
FROM (
  SELECT HOUR(start_time) as start_hour, COUNT(*) as hourly_trips
  FROM trips GROUP BY HOUR(start_time)
) hourly
ORDER BY window_trips DESC LIMIT 1;"""},

    {"question": "Which stations have a higher proportion of long trips above 45 minutes than the system average?",
     "sql": """SELECT s.station_name,
  SUM(CASE WHEN t.duration_minutes > 45 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as long_trip_pct
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
HAVING long_trip_pct > (
  SELECT SUM(CASE WHEN duration_minutes > 45 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
  FROM trips
)
ORDER BY long_trip_pct DESC;"""},

    {"question": "Find months where electric bike trips exceeded standard bike trips?",
     "sql": """SELECT month FROM (
  SELECT MONTH(start_time) as month,
    SUM(CASE WHEN bike_type = 'Electric' THEN 1 ELSE 0 END) as electric,
    SUM(CASE WHEN bike_type = 'Standard' THEN 1 ELSE 0 END) as standard
  FROM trips GROUP BY MONTH(start_time)
) monthly
WHERE electric > standard
ORDER BY month;"""},

    {"question": "What is the average trip duration percentile for each station?",
     "sql": """SELECT s.station_name,
  AVG(t.duration_minutes) as avg_duration,
  PERCENT_RANK() OVER (ORDER BY AVG(t.duration_minutes)) as duration_percentile
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY duration_percentile DESC;"""},

    {"question": "Find the top 5 stations with the most growth between first and second half of the year?",
     "sql": """SELECT s.station_name,
  first_half, second_half,
  second_half - first_half as growth
FROM (
  SELECT start_station_id,
    SUM(CASE WHEN MONTH(start_time) <= 6 THEN 1 ELSE 0 END) as first_half,
    SUM(CASE WHEN MONTH(start_time) > 6 THEN 1 ELSE 0 END) as second_half
  FROM trips GROUP BY start_station_id
) halves
JOIN stations s ON halves.start_station_id = s.station_id
ORDER BY growth DESC LIMIT 5;"""},

    {"question": "What percentage of trips occur during peak hours per station?",
     "sql": """SELECT s.station_name,
  SUM(CASE WHEN HOUR(t.start_time) BETWEEN 7 AND 9
           OR HOUR(t.start_time) BETWEEN 17 AND 19
           THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as peak_pct
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY peak_pct DESC;"""},

    {"question": "Which bike has the most trips across the most different stations?",
     "sql": """SELECT bike_id,
  COUNT(*) as total_trips,
  COUNT(DISTINCT start_station_id) as unique_stations
FROM trips
GROUP BY bike_id
ORDER BY unique_stations DESC, total_trips DESC LIMIT 1;"""},

    {"question": "What is the average gap in minutes between consecutive trips at each station?",
     "sql": """SELECT start_station_id,
  AVG(gap_minutes) as avg_gap
FROM (
  SELECT start_station_id,
    TIMESTAMPDIFF(MINUTE,
      LAG(start_time) OVER (
        PARTITION BY start_station_id ORDER BY start_time),
      start_time) as gap_minutes
  FROM trips
) gaps
WHERE gap_minutes IS NOT NULL AND gap_minutes > 0
GROUP BY start_station_id
ORDER BY avg_gap ASC;"""},

    {"question": "Which stations have never been an end station?",
     "sql": """SELECT s.station_name
FROM stations s
WHERE s.station_id NOT IN (
    SELECT DISTINCT end_station_id FROM trips
    WHERE end_station_id IS NOT NULL
)
ORDER BY s.station_name;"""},

    {"question": "Find the hour with the highest standard deviation in trip counts across all days?",
     "sql": """SELECT hour, STDDEV(daily_count) as count_stddev FROM (
  SELECT HOUR(start_time) as hour,
    DATE(start_time) as trip_date,
    COUNT(*) as daily_count
  FROM trips
  GROUP BY HOUR(start_time), DATE(start_time)
) daily_hourly
GROUP BY hour
ORDER BY count_stddev DESC LIMIT 1;"""},

    {"question": "What is the average trip duration for trips starting at the top 5 stations?",
     "sql": """SELECT AVG(duration_minutes) as avg_duration
FROM trips
WHERE start_station_id IN (
  SELECT start_station_id FROM trips
  GROUP BY start_station_id
  ORDER BY COUNT(*) DESC LIMIT 5
);"""},

    {"question": "Find stations where the most popular bike type differs between weekdays and weekends?",
     "sql": """SELECT wd.start_station_id
FROM (
  SELECT start_station_id, bike_type,
    ROW_NUMBER() OVER (
      PARTITION BY start_station_id
      ORDER BY COUNT(*) DESC) as rn
  FROM trips
  WHERE DAYOFWEEK(start_time) BETWEEN 2 AND 6
  GROUP BY start_station_id, bike_type
) wd
JOIN (
  SELECT start_station_id, bike_type,
    ROW_NUMBER() OVER (
      PARTITION BY start_station_id
      ORDER BY COUNT(*) DESC) as rn
  FROM trips
  WHERE DAYOFWEEK(start_time) IN (1,7)
  GROUP BY start_station_id, bike_type
) we ON wd.start_station_id = we.start_station_id
WHERE wd.rn = 1 AND we.rn = 1
AND wd.bike_type != we.bike_type;"""},

    {"question": "What is the trip count for each station normalized by the maximum station count?",
     "sql": """SELECT s.station_name,
  COUNT(*) as trips,
  COUNT(*) * 1.0 / MAX(COUNT(*)) OVER () as normalized_count
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY normalized_count DESC;"""},

    {"question": "Which months had both above average trips and above average temperature?",
     "sql": """SELECT t.month
FROM (
  SELECT MONTH(start_time) as month, COUNT(*) as trips
  FROM trips GROUP BY MONTH(start_time)
  HAVING trips > (
    SELECT AVG(monthly_trips) FROM (
      SELECT COUNT(*) as monthly_trips
      FROM trips GROUP BY MONTH(start_time)
    ) m)
) t
JOIN (
  SELECT MONTH(timestamp) as month, AVG(temp_c) as avg_temp
  FROM weather GROUP BY MONTH(timestamp)
  HAVING avg_temp > (SELECT AVG(temp_c) FROM weather)
) w ON t.month = w.month
ORDER BY t.month;"""},

    {"question": "Find stations where ridership dropped more than 30 percent in winter compared to summer?",
     "sql": """SELECT s.station_name,
  summer_trips, winter_trips,
  (summer_trips - winter_trips) * 100.0 / NULLIF(summer_trips, 0) as drop_pct
FROM (
  SELECT start_station_id,
    SUM(CASE WHEN MONTH(start_time) IN (6,7,8) THEN 1 ELSE 0 END) as summer_trips,
    SUM(CASE WHEN MONTH(start_time) IN (12,1,2) THEN 1 ELSE 0 END) as winter_trips
  FROM trips GROUP BY start_station_id
) seasonal
JOIN stations s ON seasonal.start_station_id = s.station_id
HAVING drop_pct > 30
ORDER BY drop_pct DESC;"""},

    {"question": "Which passholder type has the highest trip count on rainy days relative to sunny days?",
     "sql": """SELECT passholder_type,
  rainy_trips * 1.0 / NULLIF(sunny_trips, 0) as rainy_to_sunny_ratio
FROM (
  SELECT t.passholder_type,
    SUM(CASE WHEN w.precip_mm > 0 THEN 1 ELSE 0 END) as rainy_trips,
    SUM(CASE WHEN w.precip_mm = 0 THEN 1 ELSE 0 END) as sunny_trips
  FROM trips t
  JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
  GROUP BY t.passholder_type
) rain_stats
ORDER BY rainy_to_sunny_ratio DESC LIMIT 1;"""},

    {"question": "What is the median trip duration per station?",
     "sql": """SELECT s.station_name,
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY t.duration_minutes) as median_duration
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY median_duration DESC;"""},

    {"question": "Which station had the highest single day trip count in each month?",
     "sql": """SELECT month, station_name, max_daily_trips FROM (
  SELECT MONTH(t.start_time) as month,
    s.station_name,
    COUNT(*) as max_daily_trips,
    ROW_NUMBER() OVER (
      PARTITION BY MONTH(t.start_time)
      ORDER BY COUNT(*) DESC) as rn
  FROM trips t
  JOIN stations s ON t.start_station_id = s.station_id
  GROUP BY MONTH(t.start_time), DATE(t.start_time), s.station_name
) ranked
WHERE rn = 1
ORDER BY month;"""},

    {"question": "Find bikes that were used every month of the year?",
     "sql": """SELECT bike_id, COUNT(DISTINCT MONTH(start_time)) as active_months
FROM trips
GROUP BY bike_id
HAVING active_months = 12
ORDER BY bike_id;"""},

    {"question": "What is the average number of unique stations visited per bike?",
     "sql": """SELECT AVG(unique_stations) as avg_unique_stations
FROM (
  SELECT bike_id, COUNT(DISTINCT start_station_id) as unique_stations
  FROM trips GROUP BY bike_id
) bike_stations;"""},

    {"question": "Which station has the most consistent hourly demand across weekdays?",
     "sql": """SELECT s.station_name,
  STDDEV(hourly_count) as demand_stddev
FROM (
  SELECT start_station_id,
    HOUR(start_time) as hour,
    COUNT(*) as hourly_count
  FROM trips
  WHERE DAYOFWEEK(start_time) BETWEEN 2 AND 6
  GROUP BY start_station_id, HOUR(start_time)
) hourly
JOIN stations s ON hourly.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY demand_stddev ASC LIMIT 5;"""},

    {"question": "What is the trip duration distribution by deciles for each bike type?",
     "sql": """SELECT bike_type,
  NTILE(10) OVER (PARTITION BY bike_type ORDER BY duration_minutes) as decile,
  AVG(duration_minutes) as avg_duration
FROM trips
GROUP BY bike_type,
  NTILE(10) OVER (PARTITION BY bike_type ORDER BY duration_minutes)
ORDER BY bike_type, decile;"""},

    {"question": "Find stations that had zero trips on any day in January?",
     "sql": """SELECT DISTINCT s.station_name
FROM stations s
WHERE s.station_id NOT IN (
  SELECT DISTINCT start_station_id
  FROM trips
  WHERE MONTH(start_time) = 1
)
ORDER BY s.station_name;"""},

    {"question": "What is the week over week trip growth for the top 3 stations?",
     "sql": """SELECT station_name, week_num, weekly_trips,
  weekly_trips - LAG(weekly_trips) OVER (
    PARTITION BY station_name ORDER BY week_num) as wow_growth
FROM (
  SELECT s.station_name,
    WEEK(t.start_time) as week_num,
    COUNT(*) as weekly_trips
  FROM trips t JOIN stations s ON t.start_station_id = s.station_id
  WHERE s.station_id IN (
    SELECT start_station_id FROM trips
    GROUP BY start_station_id
    ORDER BY COUNT(*) DESC LIMIT 3
  )
  GROUP BY s.station_name, WEEK(t.start_time)
) weekly
ORDER BY station_name, week_num;"""},

    {"question": "Which station had the fastest growth in electric bike trips from Q1 to Q4?",
     "sql": """SELECT s.station_name,
  q1_trips, q4_trips,
  q4_trips - q1_trips as growth
FROM (
  SELECT start_station_id,
    SUM(CASE WHEN MONTH(start_time) IN (1,2,3) THEN 1 ELSE 0 END) as q1_trips,
    SUM(CASE WHEN MONTH(start_time) IN (10,11,12) THEN 1 ELSE 0 END) as q4_trips
  FROM trips
  WHERE bike_type = 'Electric'
  GROUP BY start_station_id
) quarterly
JOIN stations s ON quarterly.start_station_id = s.station_id
ORDER BY growth DESC LIMIT 1;"""},

    {"question": "What is the average trip duration for each combination of weather condition and bike type?",
     "sql": """SELECT w.weather_code, t.bike_type,
  AVG(t.duration_minutes) as avg_duration,
  COUNT(*) as trip_count
FROM trips t
JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
GROUP BY w.weather_code, t.bike_type
ORDER BY w.weather_code, t.bike_type;"""},

    {"question": "Find the top station for each passholder type?",
     "sql": """SELECT passholder_type, station_name, trips FROM (
  SELECT t.passholder_type,
    s.station_name,
    COUNT(*) as trips,
    ROW_NUMBER() OVER (
      PARTITION BY t.passholder_type
      ORDER BY COUNT(*) DESC) as rn
  FROM trips t JOIN stations s ON t.start_station_id = s.station_id
  GROUP BY t.passholder_type, s.station_name
) ranked
WHERE rn = 1
ORDER BY passholder_type;"""},

    {"question": "Which months show the strongest correlation between temperature and ridership?",
     "sql": """SELECT MONTH(t.start_time) as month,
  COUNT(*) as trips,
  AVG(w.temp_c) as avg_temp
FROM trips t
JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
GROUP BY MONTH(t.start_time)
ORDER BY (COUNT(*) * AVG(w.temp_c)) DESC;"""},

    {"question": "What is the rolling 7 day average of daily trips?",
     "sql": """SELECT trip_date, daily_trips,
  AVG(daily_trips) OVER (
    ORDER BY trip_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as rolling_7day_avg
FROM (
  SELECT DATE(start_time) as trip_date, COUNT(*) as daily_trips
  FROM trips GROUP BY DATE(start_time)
) daily
ORDER BY trip_date;"""},

    {"question": "Which station has the most trips during the morning commute compared to evening commute?",
     "sql": """SELECT s.station_name,
  SUM(CASE WHEN HOUR(t.start_time) BETWEEN 7 AND 9 THEN 1 ELSE 0 END) as morning_trips,
  SUM(CASE WHEN HOUR(t.start_time) BETWEEN 17 AND 19 THEN 1 ELSE 0 END) as evening_trips,
  SUM(CASE WHEN HOUR(t.start_time) BETWEEN 7 AND 9 THEN 1 ELSE 0 END) -
  SUM(CASE WHEN HOUR(t.start_time) BETWEEN 17 AND 19 THEN 1 ELSE 0 END) as morning_dominance
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY morning_dominance DESC LIMIT 1;"""},

    {"question": "Find all bikes that have trips longer than 2 standard deviations above the mean?",
     "sql": """SELECT DISTINCT bike_id
FROM trips
WHERE duration_minutes > (
  SELECT AVG(duration_minutes) + 2 * STDDEV(duration_minutes)
  FROM trips
)
ORDER BY bike_id;"""},

    {"question": "What is the average number of trips per active day for each station?",
     "sql": """SELECT s.station_name,
  COUNT(*) * 1.0 / COUNT(DISTINCT DATE(t.start_time)) as avg_trips_per_active_day
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY avg_trips_per_active_day DESC;"""},

    {"question": "Which hour has the most stable trip count across all days of the year?",
     "sql": """SELECT hour,
  STDDEV(daily_count) as stability_score
FROM (
  SELECT HOUR(start_time) as hour,
    DATE(start_time) as trip_date,
    COUNT(*) as daily_count
  FROM trips
  GROUP BY HOUR(start_time), DATE(start_time)
) hourly_daily
GROUP BY hour
ORDER BY stability_score ASC LIMIT 1;"""},

    {"question": "Find stations where walk-up trips grew month over month for at least 3 consecutive months?",
     "sql": """SELECT DISTINCT start_station_id
FROM (
  SELECT start_station_id, month, monthly_trips,
    LAG(monthly_trips, 1) OVER (PARTITION BY start_station_id ORDER BY month) as prev1,
    LAG(monthly_trips, 2) OVER (PARTITION BY start_station_id ORDER BY month) as prev2
  FROM (
    SELECT start_station_id,
      MONTH(start_time) as month,
      COUNT(*) as monthly_trips
    FROM trips
    WHERE passholder_type = 'Walk-up'
    GROUP BY start_station_id, MONTH(start_time)
  ) monthly
) growth
WHERE monthly_trips > prev1 AND prev1 > prev2;"""},

    {"question": "What is the average trip duration by temperature bucket?",
     "sql": """SELECT
  CASE
    WHEN w.temp_c < 10 THEN 'Cold (< 10C)'
    WHEN w.temp_c BETWEEN 10 AND 20 THEN 'Mild (10-20C)'
    WHEN w.temp_c BETWEEN 20 AND 30 THEN 'Warm (20-30C)'
    ELSE 'Hot (> 30C)'
  END as temp_bucket,
  AVG(t.duration_minutes) as avg_duration,
  COUNT(*) as trip_count
FROM trips t
JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
GROUP BY temp_bucket
ORDER BY avg_duration DESC;"""},

    {"question": "Which station pairs have trips in both directions with similar counts?",
     "sql": """SELECT a.start_station_id, a.end_station_id,
  a.trips as forward_trips, b.trips as reverse_trips,
  ABS(a.trips - b.trips) as difference
FROM (
  SELECT start_station_id, end_station_id, COUNT(*) as trips
  FROM trips WHERE start_station_id != end_station_id
  GROUP BY start_station_id, end_station_id
) a
JOIN (
  SELECT start_station_id, end_station_id, COUNT(*) as trips
  FROM trips WHERE start_station_id != end_station_id
  GROUP BY start_station_id, end_station_id
) b ON a.start_station_id = b.end_station_id
   AND a.end_station_id = b.start_station_id
WHERE a.start_station_id < a.end_station_id
ORDER BY difference ASC LIMIT 10;"""},

    {"question": "What is the longest streak of consecutive days with more than 100 trips?",
     "sql": """SELECT MAX(streak) as longest_streak
FROM (
  SELECT COUNT(*) as streak
  FROM (
    SELECT trip_date,
      DATE_SUB(trip_date, INTERVAL
        ROW_NUMBER() OVER (ORDER BY trip_date) DAY) as grp
    FROM (
      SELECT DATE(start_time) as trip_date, COUNT(*) as daily_trips
      FROM trips GROUP BY DATE(start_time)
      HAVING daily_trips > 100
    ) active_days
  ) grouped
  GROUP BY grp
) streaks;"""},

    {"question": "Find the station with the highest trip density relative to its geographic neighbors?",
     "sql": """SELECT s1.station_name,
  COUNT(DISTINCT t.trip_id) as trips,
  COUNT(DISTINCT s2.station_id) as nearby_stations
FROM trips t
JOIN stations s1 ON t.start_station_id = s1.station_id
JOIN stations s2 ON s1.station_id != s2.station_id
  AND ABS(s1.lat - s2.lat) < 0.01
  AND ABS(s1.lon - s2.lon) < 0.01
GROUP BY s1.station_name
ORDER BY trips * 1.0 / NULLIF(nearby_stations, 0) DESC LIMIT 1;"""},

    {"question": "What is the average time between a station's busiest and quietest hour?",
     "sql": """SELECT s.station_name,
  MAX(CASE WHEN rn_desc = 1 THEN hour END) as peak_hour,
  MAX(CASE WHEN rn_asc = 1 THEN hour END) as quiet_hour
FROM (
  SELECT start_station_id, hour, hourly_trips,
    ROW_NUMBER() OVER (PARTITION BY start_station_id ORDER BY hourly_trips DESC) as rn_desc,
    ROW_NUMBER() OVER (PARTITION BY start_station_id ORDER BY hourly_trips ASC) as rn_asc
  FROM (
    SELECT start_station_id,
      HOUR(start_time) as hour,
      COUNT(*) as hourly_trips
    FROM trips GROUP BY start_station_id, HOUR(start_time)
  ) hourly
) ranked
JOIN stations s ON ranked.start_station_id = s.station_id
WHERE rn_desc = 1 OR rn_asc = 1
GROUP BY s.station_name
ORDER BY s.station_name;"""},

    {"question": "Which stations show a bimodal demand pattern with peaks in both morning and evening?",
     "sql": """SELECT s.station_name,
  morning_trips, evening_trips,
  other_trips
FROM (
  SELECT start_station_id,
    SUM(CASE WHEN HOUR(start_time) BETWEEN 7 AND 9 THEN 1 ELSE 0 END) as morning_trips,
    SUM(CASE WHEN HOUR(start_time) BETWEEN 17 AND 19 THEN 1 ELSE 0 END) as evening_trips,
    SUM(CASE WHEN HOUR(start_time) NOT BETWEEN 7 AND 19 THEN 1 ELSE 0 END) as other_trips,
    COUNT(*) as total_trips
  FROM trips GROUP BY start_station_id
) demand
JOIN stations s ON demand.start_station_id = s.station_id
WHERE morning_trips > total_trips * 0.15
  AND evening_trips > total_trips * 0.15
ORDER BY (morning_trips + evening_trips) DESC;"""},

    {"question": "Find the top 5 most improved stations by trip count from first to last month?",
     "sql": """SELECT s.station_name,
  first_month_trips, last_month_trips,
  last_month_trips - first_month_trips as improvement
FROM (
  SELECT start_station_id,
    SUM(CASE WHEN MONTH(start_time) = 1 THEN 1 ELSE 0 END) as first_month_trips,
    SUM(CASE WHEN MONTH(start_time) = 12 THEN 1 ELSE 0 END) as last_month_trips
  FROM trips GROUP BY start_station_id
) monthly
JOIN stations s ON monthly.start_station_id = s.station_id
ORDER BY improvement DESC LIMIT 5;"""},

    {"question": "What is the average trip duration trend over the months of the year?",
     "sql": """SELECT MONTH(start_time) as month,
  AVG(duration_minutes) as avg_duration,
  AVG(duration_minutes) - LAG(AVG(duration_minutes)) OVER (ORDER BY MONTH(start_time)) as mom_change
FROM trips
GROUP BY MONTH(start_time)
ORDER BY month;"""},

    {"question": "Which station has the best balance of incoming and outgoing trips?",
     "sql": """SELECT s.station_name,
  outgoing, incoming,
  ABS(outgoing - incoming) as imbalance
FROM (
  SELECT station_id,
    (SELECT COUNT(*) FROM trips WHERE start_station_id = stations.station_id) as outgoing,
    (SELECT COUNT(*) FROM trips WHERE end_station_id = stations.station_id) as incoming
  FROM stations
) balance
JOIN stations s ON balance.station_id = s.station_id
ORDER BY imbalance ASC LIMIT 5;"""},

]

# Combine with original pairs
bikeshare_pairs = bikeshare_pairs + hard_pairs_remaining

# Final count
easy_count  = sum(1 for p in bikeshare_pairs[:100] if p)
med_count   = sum(1 for p in bikeshare_pairs[100:200] if p)
hard_count  = len(bikeshare_pairs) - 200

print(f" Total bike-share pairs: {len(bikeshare_pairs)}")
print(f"   Easy   : 100")
print(f"   Medium : 100")
print(f"   Hard   : {hard_count}")
print(f"\n Dataset complete and ready for formatting!")

 Total bike-share pairs: 276
   Easy   : 100
   Medium : 100
   Hard   : 76

 Dataset complete and ready for formatting!


In [37]:
# Final 24 hard pairs to reach 300 total
final_hard_pairs = [

    {"question": "Which station has the highest trip count on public holidays?",
     "sql": """SELECT s.station_name, COUNT(*) as holiday_trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
WHERE DATE(t.start_time) IN (
  '2024-01-01','2024-01-15','2024-02-19','2024-05-27',
  '2024-07-04','2024-09-02','2024-11-11','2024-11-28','2024-12-25'
)
GROUP BY s.station_name
ORDER BY holiday_trips DESC LIMIT 1;"""},

    {"question": "What is the average trip duration on holidays vs regular days?",
     "sql": """SELECT
  CASE WHEN DATE(start_time) IN (
    '2024-01-01','2024-01-15','2024-02-19','2024-05-27',
    '2024-07-04','2024-09-02','2024-11-11','2024-11-28','2024-12-25'
  ) THEN 'Holiday' ELSE 'Regular' END as day_type,
  AVG(duration_minutes) as avg_duration,
  COUNT(*) as total_trips
FROM trips
GROUP BY day_type;"""},

    {"question": "Find stations where the busiest hour shifts between summer and winter?",
     "sql": """SELECT s.station_name,
  MAX(CASE WHEN season = 'Summer' AND rn = 1 THEN hour END) as summer_peak,
  MAX(CASE WHEN season = 'Winter' AND rn = 1 THEN hour END) as winter_peak
FROM (
  SELECT start_station_id,
    CASE WHEN MONTH(start_time) IN (6,7,8) THEN 'Summer'
         WHEN MONTH(start_time) IN (12,1,2) THEN 'Winter' END as season,
    HOUR(start_time) as hour,
    COUNT(*) as trips,
    ROW_NUMBER() OVER (
      PARTITION BY start_station_id,
      CASE WHEN MONTH(start_time) IN (6,7,8) THEN 'Summer'
           WHEN MONTH(start_time) IN (12,1,2) THEN 'Winter' END
      ORDER BY COUNT(*) DESC) as rn
  FROM trips
  WHERE MONTH(start_time) IN (1,2,6,7,8,12)
  GROUP BY start_station_id, season, HOUR(start_time)
) seasonal_peaks
JOIN stations s ON seasonal_peaks.start_station_id = s.station_id
WHERE rn = 1
GROUP BY s.station_name
HAVING summer_peak != winter_peak
ORDER BY s.station_name;"""},

    {"question": "What is the trip count per station per hour ranked within each hour?",
     "sql": """SELECT s.station_name,
  HOUR(t.start_time) as hour,
  COUNT(*) as trips,
  RANK() OVER (
    PARTITION BY HOUR(t.start_time)
    ORDER BY COUNT(*) DESC) as hourly_rank
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name, HOUR(t.start_time)
ORDER BY hour, hourly_rank;"""},

    {"question": "Which bikes were used on the most different days?",
     "sql": """SELECT bike_id,
  COUNT(DISTINCT DATE(start_time)) as active_days,
  COUNT(*) as total_trips
FROM trips
GROUP BY bike_id
ORDER BY active_days DESC LIMIT 10;"""},

    {"question": "What is the percentage change in trips between each consecutive month?",
     "sql": """SELECT month,
  monthly_trips,
  LAG(monthly_trips) OVER (ORDER BY month) as prev_trips,
  ROUND((monthly_trips - LAG(monthly_trips) OVER (ORDER BY month)) * 100.0 /
    NULLIF(LAG(monthly_trips) OVER (ORDER BY month), 0), 2) as pct_change
FROM (
  SELECT MONTH(start_time) as month, COUNT(*) as monthly_trips
  FROM trips GROUP BY MONTH(start_time)
) monthly
ORDER BY month;"""},

    {"question": "Find the station with the most diverse passholder type usage?",
     "sql": """SELECT s.station_name,
  COUNT(DISTINCT t.passholder_type) as passholder_diversity,
  COUNT(*) as total_trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY passholder_diversity DESC, total_trips DESC LIMIT 1;"""},

    {"question": "What is the average trip duration for each hour split by rainy and non rainy conditions?",
     "sql": """SELECT HOUR(t.start_time) as hour,
  AVG(CASE WHEN w.precip_mm > 0 THEN t.duration_minutes END) as rainy_avg,
  AVG(CASE WHEN w.precip_mm = 0 THEN t.duration_minutes END) as dry_avg
FROM trips t
JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
GROUP BY HOUR(t.start_time)
ORDER BY hour;"""},

    {"question": "Which station has the highest proportion of trips during off peak hours?",
     "sql": """SELECT s.station_name,
  SUM(CASE WHEN HOUR(t.start_time) NOT BETWEEN 7 AND 9
           AND HOUR(t.start_time) NOT BETWEEN 17 AND 19
           THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as offpeak_pct
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY offpeak_pct DESC LIMIT 5;"""},

    {"question": "Find the month with the most balanced ridership across all stations?",
     "sql": """SELECT month, STDDEV(station_trips) as balance_score FROM (
  SELECT MONTH(start_time) as month,
    start_station_id,
    COUNT(*) as station_trips
  FROM trips
  GROUP BY MONTH(start_time), start_station_id
) monthly_station
GROUP BY month
ORDER BY balance_score ASC LIMIT 1;"""},

    {"question": "What is the average number of trips per station on rainy days?",
     "sql": """SELECT AVG(station_trips) as avg_trips_per_station
FROM (
  SELECT t.start_station_id, COUNT(*) as station_trips
  FROM trips t
  JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp)
  WHERE w.precip_mm > 0
  GROUP BY t.start_station_id
) rainy_station_trips;"""},

    {"question": "Which station pair has the highest combined trip count in both directions?",
     "sql": """SELECT
  LEAST(start_station_id, end_station_id) as station_a,
  GREATEST(start_station_id, end_station_id) as station_b,
  COUNT(*) as total_trips
FROM trips
WHERE start_station_id != end_station_id
GROUP BY
  LEAST(start_station_id, end_station_id),
  GREATEST(start_station_id, end_station_id)
ORDER BY total_trips DESC LIMIT 1;"""},

    {"question": "What is the average trip duration for the top 10 percent longest trips per station?",
     "sql": """SELECT s.station_name, AVG(duration_minutes) as top10_avg
FROM (
  SELECT start_station_id, duration_minutes,
    NTILE(10) OVER (
      PARTITION BY start_station_id
      ORDER BY duration_minutes DESC) as duration_decile
  FROM trips
) deciled
JOIN stations s ON deciled.start_station_id = s.station_id
WHERE duration_decile = 1
GROUP BY s.station_name
ORDER BY top10_avg DESC;"""},

    {"question": "Find stations where monthly pass trips grew every quarter?",
     "sql": """SELECT start_station_id FROM (
  SELECT start_station_id,
    SUM(CASE WHEN MONTH(start_time) IN (1,2,3) THEN 1 ELSE 0 END) as q1,
    SUM(CASE WHEN MONTH(start_time) IN (4,5,6) THEN 1 ELSE 0 END) as q2,
    SUM(CASE WHEN MONTH(start_time) IN (7,8,9) THEN 1 ELSE 0 END) as q3,
    SUM(CASE WHEN MONTH(start_time) IN (10,11,12) THEN 1 ELSE 0 END) as q4
  FROM trips
  WHERE passholder_type = 'Monthly Pass'
  GROUP BY start_station_id
) quarterly
WHERE q2 > q1 AND q3 > q2 AND q4 > q3;"""},

    {"question": "What is the average daily trip count per station in each season?",
     "sql": """SELECT s.station_name, season,
  AVG(daily_trips) as avg_daily
FROM (
  SELECT start_station_id,
    DATE(start_time) as trip_date,
    CASE
      WHEN MONTH(start_time) IN (3,4,5) THEN 'Spring'
      WHEN MONTH(start_time) IN (6,7,8) THEN 'Summer'
      WHEN MONTH(start_time) IN (9,10,11) THEN 'Fall'
      ELSE 'Winter'
    END as season,
    COUNT(*) as daily_trips
  FROM trips
  GROUP BY start_station_id, DATE(start_time), season
) daily
JOIN stations s ON daily.start_station_id = s.station_id
GROUP BY s.station_name, season
ORDER BY s.station_name, avg_daily DESC;"""},

    {"question": "Which station has the most trips within 1 kilometer of the city center at lat 34.05 lon minus 118.25?",
     "sql": """SELECT s.station_name, COUNT(*) as trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
WHERE (
  6371 * ACOS(
    COS(RADIANS(34.05)) * COS(RADIANS(s.lat)) *
    COS(RADIANS(s.lon) - RADIANS(-118.25)) +
    SIN(RADIANS(34.05)) * SIN(RADIANS(s.lat))
  )
) < 1
GROUP BY s.station_name
ORDER BY trips DESC LIMIT 1;"""},

    {"question": "Find bikes with above average trips in both summer and winter?",
     "sql": """SELECT bike_id
FROM (
  SELECT bike_id,
    SUM(CASE WHEN MONTH(start_time) IN (6,7,8) THEN 1 ELSE 0 END) as summer_trips,
    SUM(CASE WHEN MONTH(start_time) IN (12,1,2) THEN 1 ELSE 0 END) as winter_trips
  FROM trips GROUP BY bike_id
) seasonal
WHERE summer_trips > (
    SELECT AVG(s) FROM (
      SELECT SUM(CASE WHEN MONTH(start_time) IN (6,7,8) THEN 1 ELSE 0 END) as s
      FROM trips GROUP BY bike_id) x)
AND winter_trips > (
    SELECT AVG(w) FROM (
      SELECT SUM(CASE WHEN MONTH(start_time) IN (12,1,2) THEN 1 ELSE 0 END) as w
      FROM trips GROUP BY bike_id) x);"""},

    {"question": "What is the standard deviation of daily trips for each station?",
     "sql": """SELECT s.station_name,
  STDDEV(daily_trips) as daily_stddev,
  AVG(daily_trips) as daily_avg
FROM (
  SELECT start_station_id,
    DATE(start_time) as trip_date,
    COUNT(*) as daily_trips
  FROM trips
  GROUP BY start_station_id, DATE(start_time)
) daily
JOIN stations s ON daily.start_station_id = s.station_id
GROUP BY s.station_name
ORDER BY daily_stddev DESC;"""},

    {"question": "Which hour of the day generates the most revenue assuming walk-up trips cost more?",
     "sql": """SELECT HOUR(start_time) as hour,
  SUM(CASE WHEN passholder_type = 'Walk-up' THEN 3.5
           WHEN passholder_type = 'Monthly Pass' THEN 1.0
           ELSE 1.5 END) as estimated_revenue
FROM trips
GROUP BY HOUR(start_time)
ORDER BY estimated_revenue DESC LIMIT 1;"""},

    {"question": "Find stations with a significantly different weekday vs weekend average duration?",
     "sql": """SELECT s.station_name,
  weekday_avg, weekend_avg,
  ABS(weekday_avg - weekend_avg) as duration_difference
FROM (
  SELECT start_station_id,
    AVG(CASE WHEN DAYOFWEEK(start_time) BETWEEN 2 AND 6
        THEN duration_minutes END) as weekday_avg,
    AVG(CASE WHEN DAYOFWEEK(start_time) IN (1,7)
        THEN duration_minutes END) as weekend_avg
  FROM trips GROUP BY start_station_id
) avgs
JOIN stations s ON avgs.start_station_id = s.station_id
HAVING duration_difference > 10
ORDER BY duration_difference DESC;"""},

    {"question": "What is the trip count by bike type for each quarter?",
     "sql": """SELECT bike_type,
  SUM(CASE WHEN MONTH(start_time) IN (1,2,3) THEN 1 ELSE 0 END) as Q1,
  SUM(CASE WHEN MONTH(start_time) IN (4,5,6) THEN 1 ELSE 0 END) as Q2,
  SUM(CASE WHEN MONTH(start_time) IN (7,8,9) THEN 1 ELSE 0 END) as Q3,
  SUM(CASE WHEN MONTH(start_time) IN (10,11,12) THEN 1 ELSE 0 END) as Q4
FROM trips
GROUP BY bike_type;"""},

    {"question": "Which station had the most trips on the hottest day of the year?",
     "sql": """SELECT s.station_name, COUNT(*) as trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
WHERE DATE(t.start_time) = (
  SELECT DATE(timestamp) FROM weather
  ORDER BY temp_c DESC LIMIT 1
)
GROUP BY s.station_name
ORDER BY trips DESC LIMIT 1;"""},

    {"question": "Find the top 3 routes with the highest average trip duration?",
     "sql": """SELECT
  s1.station_name as start_station,
  s2.station_name as end_station,
  AVG(t.duration_minutes) as avg_duration,
  COUNT(*) as trip_count
FROM trips t
JOIN stations s1 ON t.start_station_id = s1.station_id
JOIN stations s2 ON t.end_station_id = s2.station_id
WHERE t.start_station_id != t.end_station_id
GROUP BY s1.station_name, s2.station_name
HAVING trip_count >= 10
ORDER BY avg_duration DESC LIMIT 3;"""},

    {"question": "What is the overall system utilization rate by hour defined as trips divided by total stations?",
     "sql": """SELECT HOUR(start_time) as hour,
  COUNT(*) as total_trips,
  (SELECT COUNT(*) FROM stations) as total_stations,
  COUNT(*) * 1.0 / (SELECT COUNT(*) FROM stations) as utilization_rate
FROM trips
GROUP BY HOUR(start_time)
ORDER BY hour;"""},

]

# Add to existing pairs
bikeshare_pairs = bikeshare_pairs + final_hard_pairs

print(f" FINAL total bike-share pairs: {len(bikeshare_pairs)}")
print(f"   Easy   : 100")
print(f"   Medium : 100")
print(f"   Hard   : {len(bikeshare_pairs) - 200}")
print(f"\n 300 pairs complete! Ready to format and combine with Spider!")

 FINAL total bike-share pairs: 300
   Easy   : 100
   Medium : 100
   Hard   : 100

 300 pairs complete! Ready to format and combine with Spider!


In [39]:
PROMPT_TEMPLATE = """### Instruction:
You are an expert SQL assistant. 
Given a database schema and a natural language question, write a correct SQL query. 
Only output the SQL query, no explanation.

### Schema:
{schema}

### Question:
{question}

### SQL:
{sql}"""

# Format all 300 bike-share pairs
bike_formatted = []
for pair in bikeshare_pairs:
    bike_formatted.append({
        "text": PROMPT_TEMPLATE.format(
            schema=BIKESHARE_SCHEMA,
            question=pair['question'],
            sql=pair['sql']
        ),
        "db_id"   : "bikeshare_la",
        "question": pair['question'],
        "sql"     : pair['sql'],
        "source"  : "bikeshare"
    })

print(f" Bike-share pairs formatted: {len(bike_formatted)}")
print("\n SAMPLE FORMATTED EXAMPLE ")
print(bike_formatted[0]['text'])

 Bike-share pairs formatted: 300

 SAMPLE FORMATTED EXAMPLE 
### Instruction:
You are an expert SQL assistant. 
Given a database schema and a natural language question, write a correct SQL query. 
Only output the SQL query, no explanation.

### Schema:
CREATE TABLE trips (
  trip_id INTEGER,
  start_time TIMESTAMP,
  end_time TIMESTAMP,
  duration_minutes FLOAT,
  start_station_id VARCHAR,
  end_station_id VARCHAR,
  bike_id VARCHAR,
  bike_type VARCHAR,
  passholder_type VARCHAR
);

CREATE TABLE stations (
  station_id VARCHAR,
  station_name VARCHAR,
  lat FLOAT,
  lon FLOAT
);

CREATE TABLE weather (
  timestamp TIMESTAMP,
  temp_c FLOAT,
  precip_mm FLOAT,
  wind_speed_mps FLOAT,
  rel_humidity FLOAT,
  pressure_hpa FLOAT,
  weather_code INTEGER
);

### Question:
How many trips were taken in total?

### SQL:
SELECT COUNT(*) FROM trips;


In [40]:
# Check all required variables are in memory
checks = {
    'spider_formatted' : 'spider_formatted' in globals(),
    'bike_formatted'   : 'bike_formatted' in globals(),
    'bikeshare_pairs'  : 'bikeshare_pairs' in globals(),
    'schema_map'       : 'schema_map' in globals(),
}

print("Variable check:")
for var, exists in checks.items():
    status = "yes" if exists else "MISSING"
    print(f"  {var:25} : {status}")

# Show counts if they exist
if checks['spider_formatted']:
    print(f"\n  spider_formatted count : {len(spider_formatted):,}")
if checks['bike_formatted']:
    print(f"  bike_formatted count   : {len(bike_formatted):,}")

Variable check:
  spider_formatted          : yes
  bike_formatted            : yes
  bikeshare_pairs           : yes
  schema_map                : yes

  spider_formatted count : 7,000
  bike_formatted count   : 300


In [41]:
#combining spider + bikeshare
from sklearn.model_selection import train_test_split

# Combine both datasets
all_data = spider_formatted + bike_formatted

print(f"Spider examples     : {len(spider_formatted):,}")
print(f"Bike-share examples : {len(bike_formatted):,}")
print(f"Total combined      : {len(all_data):,}")

# Shuffle and split 90/10 train/val
train_data, val_data = train_test_split(
    all_data,
    test_size=0.1,
    random_state=42
)

print(f"\nTrain split : {len(train_data):,}")
print(f"Val split   : {len(val_data):,}")

# Check source distribution
train_sources = [d['source'] for d in train_data]
val_sources   = [d['source'] for d in val_data]

print(f"\nSource distribution in TRAIN:")
print(f"  spider    : {train_sources.count('spider'):,}")
print(f"  bikeshare : {train_sources.count('bikeshare'):,}")

print(f"\nSource distribution in VAL:")
print(f"  spider    : {val_sources.count('spider'):,}")
print(f"  bikeshare : {val_sources.count('bikeshare'):,}")

Spider examples     : 7,000
Bike-share examples : 300
Total combined      : 7,300

Train split : 6,570
Val split   : 730

Source distribution in TRAIN:
  spider    : 6,306
  bikeshare : 264

Source distribution in VAL:
  spider    : 694
  bikeshare : 36


In [50]:
from datasets import Dataset, DatasetDict

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)

dataset = DatasetDict({
    "train"     : train_dataset,
    "validation": val_dataset
})

#  username 
DATASET_NAME = "Your Dataset Folder"

print(f"Pushing to HuggingFace Hub...")
print(f"Dataset name : {DATASET_NAME}")

dataset.push_to_hub(
    DATASET_NAME,
    private=False,
    token=HF_TOKEN
)

print(f"\n Dataset pushed successfully!")
print(f" View at: https://huggingface.co/datasets/{DATASET_NAME}")

Pushing to HuggingFace Hub...
Dataset name : Krupa1420/text2sqldata


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/529 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.



 Dataset pushed successfully!
 View at: https://huggingface.co/datasets/Krupa1420/text2sqldata


In [51]:
#Verifying
from datasets import load_dataset

DATASET_NAME = "Your DatasetFolder"

print("Reloading from HuggingFace Hub to verify...")
verify_ds = load_dataset(DATASET_NAME)

print("\n Verification results:")
print(f"  Train rows      : {len(verify_ds['train']):,}")
print(f"  Validation rows : {len(verify_ds['validation']):,}")
print(f"  Columns         : {verify_ds['train'].column_names}")

sources = verify_ds['train']['source']
print(f"\n  Spider in train    : {sources.count('spider'):,}")
print(f"  Bikeshare in train : {sources.count('bikeshare'):,}")

print("\n--- SAMPLE FROM HUGGINGFACE ---")
print(verify_ds['train'][0]['text'][:400])
print("...")
print(f"\n Phase 2 COMPLETE!")
print(f" Dataset live at: https://huggingface.co/datasets/{DATASET_NAME}")

Reloading from HuggingFace Hub to verify...


Generating train split:   0%|          | 0/6570 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/730 [00:00<?, ? examples/s]


 Verification results:
  Train rows      : 6,570
  Validation rows : 730
  Columns         : ['text', 'db_id', 'question', 'sql', 'source']

  Spider in train    : 6,306
  Bikeshare in train : 264

--- SAMPLE FROM HUGGINGFACE ---
### Instruction:
You are an expert SQL assistant. Given a database schema and a natural language question, write a correct SQL query. Only output the SQL query, no explanation.

### Schema:
CREATE TABLE Physician (
  EmployeeID number,
  Name text,
  Position text,
  SSN number
);

CREATE TABLE Department (
  DepartmentID number,
  Name text,
  Head number
);

CREATE TABLE Affiliated_With (
  Phys
...

 Phase 2 COMPLETE!
 Dataset live at: https://huggingface.co/datasets/Krupa1420/text2sqldata
